# Multi-TPC fixed-configuration ablations


In [1]:
# Configuration
# Standalone; reuses hash-verified caches from the seven-group notebook when present.
from pathlib import Path
from functools import lru_cache
from contextlib import contextmanager
from itertools import product
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score
)
from sklearn.preprocessing import StandardScaler
from IPython.display import display
import contextvars
import gc
import hashlib
import importlib.metadata
import inspect
import json
import math
import os
import pickle
import random
import re
import tempfile
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
import torch

if globals().get('_FRESH_KERNEL_SENTINEL', False):
    raise RuntimeError('Restart the kernel before running this notebook from the top.')
_FRESH_KERNEL_SENTINEL = True

SEED = 1212
FRAME_HZ = 60.0
TARGET_SR = 16000
EVENT_TOLERANCES = (0.5, 0.7, 1.0)

# Artifact keys from the executed run, mapped to descriptive constants.
METHOD_FULL = 'R12_full'
METHOD_STAGE1_ONLY = 'R12_cheap_only'
METHOD_AUDIO_ONLY = 'R12_audio_only'
METHOD_MINUS_GAZE = 'R12_minus_gaze'
METHOD_MINUS_MOTION = 'R12_minus_motion'
METHOD_MINUS_PROSODY = 'R12_minus_prosody'
METHOD_MINUS_WAVLM = 'R12_minus_wavlm'
METHOD_MINUS_ECAPA = 'R12_minus_ecapa'

PRIMARY_EVENT_TOLERANCE = 1.0
N_BOOTSTRAP = 10000

# Keep the base feature/audio contract aligned with the seven-group evaluation.
BASE_PROTOCOL_ID = 'r12-posthoc-fully-nested-logo-v3-report-complete'
IMPLEMENTATION_SHA256 = 'a5d27a8d810e6319a5ab9ca86869a91e2f7dcb9f9d4b4bf5f57a2592b97d57b0'
ABLATION_PROTOCOL = 'r12-fixed-configuration-ablation-7fold-v1'
ABLATION_REPORTING_REV = '2026-09-08-cell-audited-v2'
ABLATION_IMPLEMENTATION_REV = 'remove-one-component-from-final-r12-with-full-only-hyperparameter-selection-v2-audit-hardened'

STAGE1_MODEL_CONFIGS = (
    {'max_leaf_nodes': 15, 'learning_rate': 0.04, 'max_iter': 250, 'l2_regularization': 2.0},
    {'max_leaf_nodes': 31, 'learning_rate': 0.03, 'max_iter': 350, 'l2_regularization': 3.0},
    {'max_leaf_nodes': 47, 'learning_rate': 0.02, 'max_iter': 500, 'l2_regularization': 4.0},
)
STAGE2_REGULARIZATION_GRID = (0.1, 0.3, 1.0)
NMS_GRID_S = (0.3, 0.5, 0.7, 0.9, 1.1, 1.4)
OFFSET_GRID_S = (-0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3)
SCORE_QUANTILES = (0.40, 0.50, 0.60, 0.70, 0.78, 0.84, 0.88, 0.92, 0.95, 0.97)
DEFAULT_CANDIDATE_THRESHOLD = 0.30
DEFAULT_CANDIDATE_REFRACTORY_S = 0.50

WAVLM_MODEL_ID = 'microsoft/wavlm-large'
WAVLM_MODEL_REVISION = 'c1423ed94bb01d80a3f5ce5bc39f6026a0f4828c'
WAVLM_CONTEXT_S = 4.0
WAVLM_RECENT_CONTEXT_S = 0.5
WAVLM_PROJECTION_DIM = 48
ECAPA_MODEL_ID = 'speechbrain/spkrec-ecapa-voxceleb'
ECAPA_SEGMENT_S = 1.25
ECAPA_MIN_SEGMENT_S = 0.25
# Names kept for cache compatibility with the seven-group notebook.
AUDIO_STACK_ID = BASE_PROTOCOL_ID

ALIGNMENT_SEARCH_MAX_S = 2.0
ALIGNMENT_WARN_ABS_S = 0.5
ALIGNMENT_BOUNDARY_MARGIN_S = 0.05
ALIGNMENT_CONTRACT_REV = 'offset-only-exact-60hz-v17.4-absolute-final-r3'
MODALITY_ROW_MISMATCH_MAX_S = 1.0
ANNOTATION_SCHEMA_OVERRIDE = None
PROSODY_COLUMN_OVERRIDE = None
NATIVE_NTT_MIN_GAP_S = 0.2
ITT_RESOLUTION_S = 1.0
ITT_MIN_SUCCESSFUL_FLOOR_S = 0.3
CUDA_MIN_FREE_GB_FOR_MAIN = float(os.environ.get('CUDA_MIN_FREE_GB_FOR_MAIN', '4.0'))
FORCE_MAIN_CPU = bool(int(os.environ.get('FORCE_MAIN_CPU', '0')))
RESOURCE_PROFILE = 'single_gpu_sequential_ablation'

_root_env = os.environ.get('MULTITPC_ROOT', '').strip()
_root_candidates = []
if _root_env:
    _root_candidates.append(Path(_root_env).expanduser())
for _base in [Path.cwd(), *Path.cwd().parents]:
    _root_candidates.extend([
        _base / 'data' / 'Multi-TPC',
        _base / 'Multi-TPC',
    ])
MULTITPC_ROOT = next((p.resolve() for p in _root_candidates if p.exists()), None)
if MULTITPC_ROOT is None:
    raise FileNotFoundError(
        "Set MULTITPC_ROOT to the Multi-TPC dataset/project root before running."
    )

ABLATION_ROOT = MULTITPC_ROOT / 'runs' / 'multitpc_fixed_configuration_ablation'
CACHE_ROOT = ABLATION_ROOT / 'cache'
DETERMINISTIC_CACHE_ROOT = CACHE_ROOT / 'deterministic_alignment'
BASE_MAIN_ROOT = MULTITPC_ROOT / 'runs' / 'multitpc_seven_group_nested'
BASE_MAIN_CACHE = BASE_MAIN_ROOT / 'cache'

# Optional cache reuse is safe because every reused item is contract/hash checked.
_shared_audio_env = os.environ.get('MULTITPC_SHARED_AUDIO_CACHE', os.environ.get('R12_SHARED_AUDIO_CACHE_ROOT', '')).strip()
_default_shared_audio = BASE_MAIN_CACHE / 'wavlm_ecapa'
AUDIO_FEATURE_CACHE_ROOT = Path(_shared_audio_env).expanduser().resolve() if _shared_audio_env else (_default_shared_audio if _default_shared_audio.exists() else CACHE_ROOT / 'wavlm_ecapa')
_shared_causal_env = os.environ.get('MULTITPC_SHARED_CAUSAL_CACHE', os.environ.get('R12_SHARED_CAUSAL_CACHE_ROOT', '')).strip()
DEFAULT_CAUSAL_CACHE = Path(_shared_causal_env).expanduser().resolve() if _shared_causal_env else ((BASE_MAIN_CACHE / 'r12_causal_tables') if (BASE_MAIN_CACHE / 'r12_causal_tables').exists() else CACHE_ROOT / 'r12_causal_tables')

for _p in (ABLATION_ROOT, CACHE_ROOT, DETERMINISTIC_CACHE_ROOT, AUDIO_FEATURE_CACHE_ROOT, DEFAULT_CAUSAL_CACHE):
    _p.mkdir(parents=True, exist_ok=True)

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print('Ablation protocol:', ABLATION_PROTOCOL)
print('Multi-TPC root:', MULTITPC_ROOT)
print('Ablation output root:', ABLATION_ROOT)
print('Causal cache:', DEFAULT_CAUSAL_CACHE)
print('WavLM/ECAPA cache:', AUDIO_FEATURE_CACHE_ROOT)

Ablation protocol: r12-fixed-configuration-ablation-7fold-v1
Multi-TPC root: <MULTITPC_ROOT>
Ablation output root: <MULTITPC_ROOT>/runs/MultiTPC_R12_FIXED_CONFIGURATION_ABLATIONS_7FOLD
Causal cache: <MULTITPC_ROOT>/runs/MultiTPC_R12_FIXED_CONFIGURATION_ABLATIONS_7FOLD/cache/r12_causal_tables
WavLM/ECAPA cache: <MULTITPC_ROOT>/runs/MultiTPC_R12_FIXED_CONFIGURATION_ABLATIONS_7FOLD/cache/wavlm_ecapa


In [2]:
# Reproducibility, hashing, hardware, and resource helpers
warnings.filterwarnings('default')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def stable_seed(tag, base=SEED):
    raw=f'{int(base)}::{str(tag)}'.encode('utf8')
    return int.from_bytes(hashlib.sha256(raw).digest()[:4], 'little') & 0x7fffffff

def set_run_seed(tag, base=SEED):
    sd=stable_seed(tag,base)
    random.seed(sd); np.random.seed(sd); torch.manual_seed(sd)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(sd)
    return sd

def _support_jsonable_config(value):
    if isinstance(value, np.ndarray): return [_support_jsonable_config(x) for x in value.tolist()]
    if isinstance(value, np.generic): return _support_jsonable_config(value.item())
    if isinstance(value,float) and not math.isfinite(value): return None
    if isinstance(value, Path): return str(value)
    if isinstance(value, (list, tuple)): return [_support_jsonable_config(x) for x in value]
    if isinstance(value, dict): return {str(k): _support_jsonable_config(v) for k, v in value.items()}
    return value

def _support_sha256_json(obj, n=24):
    raw=json.dumps(obj,sort_keys=True,default=str,separators=(',',':')).encode('utf8')
    return hashlib.sha256(raw).hexdigest()[:int(n)]

def _sha256_file(path, block=1 << 20):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            b=f.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()

SUPPORT_FILE_HASH_INDEX_PATH = CACHE_ROOT / 'required_file_sha256_index.json'
try:
    _SUPPORT_FILE_HASH_INDEX = json.loads(SUPPORT_FILE_HASH_INDEX_PATH.read_text()) if SUPPORT_FILE_HASH_INDEX_PATH.exists() else {}
except Exception:
    _SUPPORT_FILE_HASH_INDEX = {}

def _support_sha256_file(path, block_bytes=8*1024*1024):
    return _sha256_file(path, block_bytes)

def _support_file_identity(path):
    p=Path(path).resolve(); st=p.stat(); key=str(p)
    cached=_SUPPORT_FILE_HASH_INDEX.get(key,{})
    same=(int(cached.get('size',-1))==int(st.st_size) and int(cached.get('mtime_ns',-1))==int(st.st_mtime_ns) and bool(cached.get('sha256')))
    sha=str(cached['sha256']) if same else _support_sha256_file(p)
    rec=dict(path=key,size=int(st.st_size),mtime_ns=int(st.st_mtime_ns),sha256=sha)
    _SUPPORT_FILE_HASH_INDEX[key]=rec
    return rec

def _support_flush_file_hash_index():
    SUPPORT_FILE_HASH_INDEX_PATH.parent.mkdir(parents=True,exist_ok=True)
    SUPPORT_FILE_HASH_INDEX_PATH.write_text(json.dumps(_SUPPORT_FILE_HASH_INDEX,indent=2,sort_keys=True))

def _support_table_fingerprint(df, columns):
    q=df.loc[:,list(columns)].copy() if len(df) else pd.DataFrame(columns=list(columns))
    order=list(columns)[:2] if len(columns)>=2 else list(columns)
    return _support_sha256_json(q.sort_values(order).to_dict('records'))

def _cuda_memory_snapshot():
    out={'available':bool(torch.cuda.is_available()),'free_gb':0.0,'total_gb':0.0}
    if not torch.cuda.is_available(): return out
    try:
        free_b,total_b=torch.cuda.mem_get_info()
        out.update(free_gb=float(free_b)/(1024**3),total_gb=float(total_b)/(1024**3))
    except Exception as exc: out['error']=repr(exc)
    return out

CUDA_START=_cuda_memory_snapshot()
_cuda_ok=(CUDA_START.get('available',False) and not FORCE_MAIN_CPU and CUDA_START.get('free_gb',0.0)>=float(CUDA_MIN_FREE_GB_FOR_MAIN))
DEVICE=torch.device('cuda' if _cuda_ok else 'cpu')

def release_resources(*objs,note=''):
    for obj in objs:
        try:
            if isinstance(obj,torch.nn.Module): obj.to('cpu')
        except Exception: pass
    gc.collect()
    if torch.cuda.is_available():
        try: torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        except Exception: pass
    if note:
        snap=_cuda_memory_snapshot(); print(f'[resource cleanup] {note}: free CUDA={snap.get("free_gb",0.0):.2f} GiB')

def ensure_min_cuda_free(stage,min_free_gb=None):
    if DEVICE.type!='cuda': return False
    need=float(CUDA_MIN_FREE_GB_FOR_MAIN if min_free_gb is None else min_free_gb)
    release_resources(note=f'before {stage}')
    snap=_cuda_memory_snapshot(); ok=snap.get('free_gb',0.0)>=need
    if not ok: print(f'WARNING: {stage} has only {snap.get("free_gb",0.0):.2f} GiB free CUDA (<{need:.2f} GiB).')
    return ok

try: torch.set_float32_matmul_precision('high')
except Exception: pass
print('Resource profile:',RESOURCE_PROFILE)
print('Device:',DEVICE)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))

Resource profile: single_gpu_sequential_r12_ablation
Device: cuda
GPU: NVIDIA GeForce RTX 4080 SUPER


In [3]:
# Dataset discovery and content fingerprint
ANNOT_RE = re.compile(r"(?P<date>\d{2}-\d{2}-\d{4})_Session_(?P<session>\d+)_audio_gaze_gback\.csv$", re.I)

def resolve_dir(*names):
    for n in names:
        p=MULTITPC_ROOT/n
        if p.exists(): return p
    return MULTITPC_ROOT/names[0]

AUDIO_DIR=resolve_dir("Audio","audio")
ANNOT_DIR=resolve_dir("AudioGazeGBack","audiogazegback")
WORD_DIR=resolve_dir("Word","word")
PROSODY_DIR=resolve_dir("Prosody","prosody")
MOCAP_DIR=resolve_dir("Mocap","mocap")
GAZE_DIR=resolve_dir("Gaze","gaze")

def session_id(date,n):
    mm,dd,yyyy=date.split("-")
    return f"mtpc_{yyyy}{mm}{dd}_s{int(n):02d}"

def discover_sessions():
    if not ANNOT_DIR.exists():
        raise FileNotFoundError(f"Missing annotation directory: {ANNOT_DIR}")
    rows=[]
    for ap in sorted(ANNOT_DIR.glob("*_audio_gaze_gback.csv")):
        m=ANNOT_RE.search(ap.name)
        if not m: continue
        date=m.group("date"); sn=int(m.group("session")); sid=session_id(date,sn)
        r=dict(session=sid,date=date,group_id=date,session_number=sn,annotation_path=str(ap),
               prosody_path=str(PROSODY_DIR/f"{date}_Session_{sn}_prosody.csv"))
        for p in (1,2,3):
            r[f"p{p}_audio_path"]=str(AUDIO_DIR/date/f"Session_{sn}_PC_{p}_audio.wav")
            r[f"p{p}_word_path"]=str(WORD_DIR/date/f"Session_{sn}_PC_{p}_words.csv")
            r[f"p{p}_mocap_path"]=str(MOCAP_DIR/date/f"Session_{sn}_PC_{p}_mocap_data.txt")
            g1=GAZE_DIR/date/f"Session_{sn}_PC_{p}_EyeTracker_data_gapfilled.txt"
            g2=GAZE_DIR/date/f"Session_{sn}_PC_{p}_EyeTracker_data.txt"
            r[f"p{p}_gaze_path"]=str(g1 if g1.exists() else g2)
        rows.append(r)
    mf=pd.DataFrame(rows).sort_values(["date","session_number"]).reset_index(drop=True)
    if mf.empty: raise RuntimeError("No Multi-TPC sessions discovered.")
    mf["group_index"]=mf["group_id"].map({g:i for i,g in enumerate(sorted(mf.group_id.unique()))})
    return mf

MANIFEST=discover_sessions()
print("Discovered sessions:",len(MANIFEST),"groups:",MANIFEST.group_id.nunique())
if len(MANIFEST)!=24:
    print(
        "NOTE: the Scientific Data paper reports 24 sessions. "
        f"This local release contains {len(MANIFEST)} complete discovered annotation sessions; "
        "the notebook uses exactly what is present and records the count."
    )
display(MANIFEST[["session","group_id","session_number"]])

DISCOVERY_AUDIT=[]
for r in MANIFEST.itertuples(index=False):
    paths=[r.annotation_path,r.prosody_path]
    paths += [getattr(r,f"p{p}_{k}_path") for p in (1,2,3) for k in ("audio","word","mocap","gaze")]
    missing=[x for x in paths if not Path(x).exists()]
    DISCOVERY_AUDIT.append(dict(session=r.session,n_missing=len(missing),missing=" | ".join(missing)))
DISCOVERY_AUDIT=pd.DataFrame(DISCOVERY_AUDIT)
display(DISCOVERY_AUDIT)
if DISCOVERY_AUDIT.n_missing.gt(0).any():
    bad=DISCOVERY_AUDIT[DISCOVERY_AUDIT.n_missing.gt(0)]
    raise FileNotFoundError(
        "The discovered Multi-TPC release is incomplete for the selected master experiment.\n"
        + bad[["session","missing"]].to_string(index=False)
    )
_SUPPORT_FINGERPRINT_PATHS=[]
for r in MANIFEST.itertuples(index=False):
    _SUPPORT_FINGERPRINT_PATHS.extend([r.annotation_path,r.prosody_path])
    for _p in (1,2,3):
        _SUPPORT_FINGERPRINT_PATHS.extend([getattr(r,f"p{_p}_audio_path"),getattr(r,f"p{_p}_word_path"),getattr(r,f"p{_p}_mocap_path"),getattr(r,f"p{_p}_gaze_path")])
SUPPORT_REQUIRED_FILE_IDENTITIES=[_support_file_identity(p) for p in sorted(set(_SUPPORT_FINGERPRINT_PATHS))]
_support_flush_file_hash_index()
MULTITPC_DATA_FINGERPRINT=_support_sha256_json(SUPPORT_REQUIRED_FILE_IDENTITIES)
print("Multi-TPC content fingerprint:",MULTITPC_DATA_FINGERPRINT,"files:",len(SUPPORT_REQUIRED_FILE_IDENTITIES))

Discovered sessions: 23 groups: 7
NOTE: the Scientific Data paper reports 24 sessions. This local release contains 23 complete discovered annotation sessions; the notebook uses exactly what is present and records the count.


,session,group_id,session_number
0,mtpc_20220128_s01,01-28-2022,1
1,mtpc_20220128_s02,01-28-2022,2
2,mtpc_20220128_s03,01-28-2022,3
3,mtpc_20220128_s04,01-28-2022,4
4,mtpc_20220128_s05,01-28-2022,5
5,mtpc_20220128_s06,01-28-2022,6
6,mtpc_20220211_s02,02-11-2022,2
7,mtpc_20220211_s03,02-11-2022,3
8,mtpc_20220304_s01,03-04-2022,1
9,mtpc_20220304_s02,03-04-2022,2


,session,n_missing,missing
0,mtpc_20220128_s01,0,
1,mtpc_20220128_s02,0,
2,mtpc_20220128_s03,0,
3,mtpc_20220128_s04,0,
4,mtpc_20220128_s05,0,
5,mtpc_20220128_s06,0,
6,mtpc_20220211_s02,0,
7,mtpc_20220211_s03,0,
8,mtpc_20220304_s01,0,
9,mtpc_20220304_s02,0,


Multi-TPC content fingerprint: 568dbda5fa6da3f1f757c5bc files: 322


In [4]:
# Annotation loader
SESSION_ROW={r.session:r._asdict() for r in MANIFEST.itertuples(index=False)}

def _normcol(c):
    return re.sub(r"[^a-z0-9]+","",str(c).lower())

def _participant_col(cols,p,kind):
    """Resolve one official participant column without depending on file column order."""
    pn=f"p{int(p)}"
    candidates=[]
    for c in cols:
        n=_normcol(c)
        if pn not in n:
            continue
        if kind=="audio" and "audio" in n:
            candidates.append(c)
        elif kind=="gaze" and "gaze" in n:
            candidates.append(c)
        elif kind=="headshake" and ("headshak" in n or "shaking" in n):
            candidates.append(c)
        elif kind=="nodding" and ("nodd" in n):
            candidates.append(c)
    return candidates[0] if len(candidates)==1 else None

@lru_cache(maxsize=64)
def load_annotation(session):
    """Load the audited native 60-Hz annotation using named columns only.

    The executed V16 audit confirmed the local release names:
      P1/P2/P3 audio, P1/P2/P3 Gaze,
      p1/p2/p3_headshaking and p1/p2/p3_nodding.
    A future release with renamed columns must be mapped explicitly through
    ANNOTATION_SCHEMA_OVERRIDE; silent positional guessing is prohibited.
    """
    row=SESSION_ROW[str(session)]
    df=pd.read_csv(row["annotation_path"])
    cols=list(df.columns)

    if ANNOTATION_SCHEMA_OVERRIDE is not None:
        sch=dict(ANNOTATION_SCHEMA_OVERRIDE)
        audio=list(sch.get("audio",[]));gaze=list(sch.get("gaze",[]));head=list(sch.get("head",[]))
    else:
        audio=[_participant_col(cols,p,"audio") for p in (1,2,3)]
        gaze=[_participant_col(cols,p,"gaze") for p in (1,2,3)]
        head=[]
        for p in (1,2,3):
            head.extend([_participant_col(cols,p,"headshake"),_participant_col(cols,p,"nodding")])

    missing=[x for x in audio+gaze+head if x is None or x not in cols]
    if missing:
        raise RuntimeError(
            f"{session}: could not resolve the audited named annotation schema. "
            f"Observed columns={cols}. Set ANNOTATION_SCHEMA_OVERRIDE explicitly if this is a renamed release."
        )
    if len(audio)!=3 or len(gaze)!=3 or len(head)!=6 or len(set(audio+gaze+head))!=12:
        raise RuntimeError(f"{session}: invalid annotation schema mapping: audio={audio}, gaze={gaze}, head={head}")

    out={
        "df":df,
        "audio":df[audio].apply(pd.to_numeric,errors="coerce").fillna(0).to_numpy(np.int16),
        "gaze":df[gaze].apply(pd.to_numeric,errors="coerce").fillna(0).to_numpy(np.int16),
        "head":df[head].apply(pd.to_numeric,errors="coerce").fillna(0).to_numpy(np.float32),
        "audio_cols":list(audio),"gaze_cols":list(gaze),"head_cols":list(head),
    }
    if out["audio"].shape[1]!=3 or out["gaze"].shape[1]!=3 or out["head"].shape[1]!=6:
        raise RuntimeError(f"{session}: annotation tensor dimensions violate the 3/3/6 contract")
    if not set(np.unique(out["audio"]).tolist()).issubset({0,1,2}):
        raise RuntimeError(f"{session}: unexpected native activity state outside {{0,1,2}}")
    if not set(np.unique(out["gaze"]).tolist()).issubset({0,1,2}):
        raise RuntimeError(f"{session}: unexpected native gaze code outside {{0,1,2}}")
    return out

AUDIT_ROWS=[]
for s in MANIFEST.session:
    a=load_annotation(s)
    states=set(np.unique(a["audio"]).tolist())
    gazes=set(np.unique(a["gaze"]).tolist())
    AUDIT_ROWS.append(dict(
        session=s,
        n_frames=len(a["audio"]),
        duration_s=len(a["audio"])/FRAME_HZ,
        audio_states=sorted(states),
        gaze_states=sorted(gazes),
        audio_cols=" | ".join(map(str,a["audio_cols"])),
        gaze_cols=" | ".join(map(str,a["gaze_cols"])),
        head_cols=" | ".join(map(str,a["head_cols"])),
    ))

ANNOT_AUDIT=pd.DataFrame(AUDIT_ROWS)
display(ANNOT_AUDIT)
assert all(set(x).issubset({0,1,2}) for x in ANNOT_AUDIT.audio_states), "Unexpected native audio-state code."
assert all(set(x).issubset({0,1,2}) for x in ANNOT_AUDIT.gaze_states), "Unexpected native gaze code."

,session,n_frames,duration_s,audio_states,gaze_states,audio_cols,gaze_cols,head_cols
0,mtpc_20220128_s01,19377,322.950000,"[0, 1, 2]","[0, 1, 2]",P1 audio | P2 audio | P3 audio,P1 Gaze | P2 Gaze | P3 Gaze,p1_headshaking | p1_nodding | p2_headshaking |...
1,mtpc_20220128_s02,33463,557.716667,"[0, 1, 2]","[0, 1, 2]",P1 audio | P2 audio | P3 audio,P1 Gaze | P2 Gaze | P3 Gaze,p1_headshaking | p1_nodding | p2_headshaking |...
2,mtpc_20220128_s03,21566,359.433333,"[0, 1, 2]","[0, 1, 2]",P1 audio | P2 audio | P3 audio,P1 Gaze | P2 Gaze | P3 Gaze,p1_headshaking | p1_nodding | p2_headshaking |...
3,mtpc_20220128_s04,36625,610.416667,"[0, 1, 2]","[0, 1, 2]",P1 audio | P2 audio | P3 audio,P1 Gaze | P2 Gaze | P3 Gaze,p1_headshaking | p1_nodding | p2_headshaking |...
4,mtpc_20220128_s05,25730,428.833333,"[0, 1, 2]","[0, 1, 2]",P1 audio | P2 audio | P3 audio,P1 Gaze | P2 Gaze | P3 Gaze,p1_headshaking | p1_nodding | p2_headshaking |...
5,mtpc_20220128_s06,19681,328.016667,"[0, 1, 2]","[0, 1, 2]",P1 audio | P2 audio | P3 audio,P1 Gaze | P2 Gaze | P3 Gaze,p1_headshaking | p1_nodding | p2_headshaking |...
6,mtpc_20220211_s02,43968,732.800000,"[0, 1, 2]","[0, 1, 2]",P1 audio | P2 audio | P3 audio,P1 Gaze | P2 Gaze | P3 Gaze,p1_headshaking | p1_nodding | p2_headshaking |...
7,mtpc_20220211_s03,68872,1147.866667,"[0, 1, 2]","[0, 1, 2]",P1 audio | P2 audio | P3 audio,P1 Gaze | P2 Gaze | P3 Gaze,p1_headshaking | p1_nodding | p2_headshaking |...
8,mtpc_20220304_s01,47308,788.466667,"[0, 1, 2]","[0, 1, 2]",P1 audio | P2 audio | P3 audio,P1 Gaze | P2 Gaze | P3 Gaze,p1_headshaking | p1_nodding | p2_headshaking |...
9,mtpc_20220304_s02,49953,832.550000,"[0, 1, 2]","[0, 1, 2]",P1 audio | P2 audio | P3 audio,P1 Gaze | P2 Gaze | P3 Gaze,p1_headshaking | p1_nodding | p2_headshaking |...


In [5]:
# Audio alignment and acoustic cache
def resample_linear(x, orig_sr, target_sr):
    x = np.asarray(x, np.float32)

    if int(orig_sr) == int(target_sr):
        return x

    if len(x) == 0:
        return np.zeros(0, dtype=np.float32)

    n = max(
        1,
        int(
            round(
                len(x)
                * float(target_sr)
                / float(orig_sr)
            )
        ),
    )

    old = np.linspace(
        0.0,
        1.0,
        len(x),
        endpoint=False,
    )

    new = np.linspace(
        0.0,
        1.0,
        n,
        endpoint=False,
    )

    return np.interp(
        new,
        old,
        x.astype(float),
    ).astype(np.float32)

@lru_cache(maxsize=16)
def read_participant_audio(session, p):
    session = str(session)
    p = int(p)

    if session not in SESSION_ROW:
        raise KeyError(
            f"Session {session!r} is not present in SESSION_ROW."
        )

    key = f"p{p}_audio_path"

    if key not in SESSION_ROW[session]:
        raise KeyError(
            f"{key!r} is missing for session {session!r}."
        )

    path = Path(
        SESSION_ROW[session][key]
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Participant audio not found: {path}"
        )

    x, sr = sf.read(
        str(path),
        dtype="float32",
        always_2d=False,
    )

    x = np.asarray(x)

    if x.ndim > 1:
        x = x.mean(axis=1)

    x = np.nan_to_num(
        x,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    ).astype(np.float32)

    if int(sr) != int(TARGET_SR):
        x = resample_linear(
            x,
            int(sr),
            int(TARGET_SR),
        )
        sr = int(TARGET_SR)

    return x, int(sr)

def _audio_frame_rms(x, sr, hz=60.0):
    """RMS on an exact `hz` clock using sample-index boundaries round(k*sr/hz).

    At 16 kHz / 60 Hz the bins necessarily alternate between 266 and 267 samples.
    Using one fixed 267-sample block would create an artificial ~59.925-Hz clock and
    cumulative drift, so fixed-length reshape is explicitly prohibited here.
    """
    x=np.asarray(x,np.float32)
    sr=float(sr);hz=float(hz)
    if len(x)==0:return np.zeros(0,dtype=np.float32)
    if not (sr>0 and hz>0):raise ValueError(f"Invalid sr/hz: sr={sr}, hz={hz}")
    n_frames=int(np.floor(len(x)*hz/sr + 1e-12))
    if n_frames<=0:return np.zeros(0,dtype=np.float32)
    edges=np.rint(np.arange(n_frames+1,dtype=np.float64)*sr/hz).astype(np.int64)
    edges=np.clip(edges,0,len(x))
    if np.any(np.diff(edges)<=0):raise RuntimeError("Exact audio-frame boundaries are not strictly increasing")
    sq=np.asarray(x,dtype=np.float64)**2
    cs=np.concatenate(([0.0],np.cumsum(sq,dtype=np.float64)))
    sums=cs[edges[1:]]-cs[edges[:-1]]
    counts=(edges[1:]-edges[:-1]).astype(np.float64)
    return np.sqrt(sums/counts + 1e-10).astype(np.float32)

AUDIO_OFFSET_PATH = (
    DETERMINISTIC_CACHE_ROOT
    / f"audio_offset_{ALIGNMENT_CONTRACT_REV}_{MULTITPC_DATA_FINGERPRINT}.csv"
)

AUDIO_OFFSET_COLUMNS = [
    "session",
    "participant",
    "lag_frames",
    "offset_s",
    "corr",
    "boundary_solution",
    "warning",
]

if AUDIO_OFFSET_PATH.exists():

    AUDIO_OFFSETS = pd.read_csv(
        AUDIO_OFFSET_PATH
    )

    for col in AUDIO_OFFSET_COLUMNS:
        if col not in AUDIO_OFFSETS.columns:
            if col in (
                "lag_frames",
                "offset_s",
                "corr",
            ):
                AUDIO_OFFSETS[col] = np.nan

            elif col in (
                "boundary_solution",
                "warning",
            ):
                AUDIO_OFFSETS[col] = False

            else:
                AUDIO_OFFSETS[col] = ""

    AUDIO_OFFSETS["session"] = (
        AUDIO_OFFSETS["session"]
        .astype(str)
    )

    AUDIO_OFFSETS["participant"] = pd.to_numeric(
        AUDIO_OFFSETS["participant"],
        errors="coerce",
    )

    AUDIO_OFFSETS["lag_frames"] = pd.to_numeric(
        AUDIO_OFFSETS["lag_frames"],
        errors="coerce",
    )

    AUDIO_OFFSETS["offset_s"] = pd.to_numeric(
        AUDIO_OFFSETS["offset_s"],
        errors="coerce",
    )

    AUDIO_OFFSETS["corr"] = pd.to_numeric(
        AUDIO_OFFSETS["corr"],
        errors="coerce",
    )

    AUDIO_OFFSETS = (
        AUDIO_OFFSETS[
            AUDIO_OFFSET_COLUMNS
        ]
        .dropna(
            subset=[
                "participant",
                "offset_s",
                "corr",
            ]
        )
        .copy()
    )

    AUDIO_OFFSETS["participant"] = (
        AUDIO_OFFSETS["participant"]
        .astype(int)
    )

else:

    AUDIO_OFFSETS = pd.DataFrame(
        columns=AUDIO_OFFSET_COLUMNS
    )

def estimate_audio_offset(
    session,
    p,
    max_abs_s=ALIGNMENT_SEARCH_MAX_S,
):
    global AUDIO_OFFSETS

    session = str(session)
    p = int(p)

    hit = AUDIO_OFFSETS[
        (
            AUDIO_OFFSETS[
                "session"
            ].astype(str)
            == session
        )
        &
        (
            AUDIO_OFFSETS[
                "participant"
            ].astype(int)
            == p
        )
    ]

    if len(hit):

        row = hit.iloc[0]

        offset_s = float(
            row["offset_s"]
        )

        corr = float(
            row["corr"]
        )

        if np.isfinite(offset_s) and np.isfinite(corr):
            return offset_s, corr

        print(
            f"WARNING: invalid cached audio offset for "
            f"{session} P{p}; recomputing."
        )

    x, sr = read_participant_audio(
        session,
        p,
    )

    env = _audio_frame_rms(
        x,
        sr,
        FRAME_HZ,
    )

    ann = load_annotation(
        session
    )

    target = (
        ann["audio"][:, p - 1]
        > 0
    ).astype(np.float32)

    if len(env) == 0 or len(target) == 0:
        print(
            f"WARNING: empty audio/alignment signal for "
            f"{session} P{p}; using zero offset."
        )
        return 0.0, 0.0

    pos = env[env > 0]

    scale = (
        float(np.median(pos))
        if len(pos)
        else 1.0
    )

    env = np.log1p(
        env / max(1e-8, scale)
    )

    env = (
        env - env.mean()
    ) / (
        env.std() + 1e-8
    )

    target = (
        target - target.mean()
    ) / (
        target.std() + 1e-8
    )

    maxlag = int(
        round(
            float(max_abs_s)
            * float(FRAME_HZ)
        )
    )

    best_lag = 0
    best_corr = -np.inf

    for lag in range(
        -maxlag,
        maxlag + 1,
        2,
    ):

        i0 = max(
            0,
            -lag,
        )

        i1 = min(
            len(target),
            len(env) - lag,
        )

        if i1 - i0 < 300:
            continue

        c = float(
            np.mean(
                target[i0:i1]
                * env[
                    i0 + lag:
                    i1 + lag
                ]
            )
        )

        if np.isfinite(c) and c > best_corr:
            best_lag = int(lag)
            best_corr = float(c)

    coarse = int(best_lag)

    for lag in range(
        max(
            -maxlag,
            coarse - 4,
        ),
        min(
            maxlag,
            coarse + 4,
        )
        + 1,
    ):

        i0 = max(
            0,
            -lag,
        )

        i1 = min(
            len(target),
            len(env) - lag,
        )

        if i1 - i0 < 300:
            continue

        c = float(
            np.mean(
                target[i0:i1]
                * env[
                    i0 + lag:
                    i1 + lag
                ]
            )
        )

        if np.isfinite(c) and c > best_corr:
            best_lag = int(lag)
            best_corr = float(c)

    if not np.isfinite(best_corr):
        print(
            f"WARNING: no valid offset correlation for "
            f"{session} P{p}; using zero offset."
        )

        best_lag = 0
        best_corr = 0.0

    lag = int(best_lag)
    corr = float(best_corr)

    offset_s = float(
        lag / float(FRAME_HZ)
    )

    boundary = bool(
        abs(
            abs(offset_s)
            - float(max_abs_s)
        )
        <= float(
            ALIGNMENT_BOUNDARY_MARGIN_S
        )
    )

    warning = bool(
        abs(offset_s)
        > float(
            ALIGNMENT_WARN_ABS_S
        )
        or boundary
    )

    row = {
        "session": session,
        "participant": p,
        "lag_frames": lag,
        "offset_s": offset_s,
        "corr": corr,
        "boundary_solution": boundary,
        "warning": warning,
    }

    AUDIO_OFFSETS = AUDIO_OFFSETS[
        ~(
            (
                AUDIO_OFFSETS[
                    "session"
                ].astype(str)
                == session
            )
            &
            (
                AUDIO_OFFSETS[
                    "participant"
                ].astype(int)
                == p
            )
        )
    ].copy()

    AUDIO_OFFSETS = pd.concat(
        [
            AUDIO_OFFSETS,
            pd.DataFrame([row]),
        ],
        ignore_index=True,
    )

    AUDIO_OFFSETS = (
        AUDIO_OFFSETS[
            AUDIO_OFFSET_COLUMNS
        ]
        .sort_values(
            [
                "session",
                "participant",
            ]
        )
        .reset_index(drop=True)
    )

    AUDIO_OFFSETS.to_csv(
        AUDIO_OFFSET_PATH,
        index=False,
    )

    return offset_s, corr

def annotation_time_to_raw_audio(
    session,
    p,
    t,
):
    offset_s, _ = estimate_audio_offset(
        session,
        p,
    )

    return (
        float(t)
        + float(offset_s)
    )

audio_audit = []

for s in MANIFEST["session"].astype(str):

    for p in (1, 2, 3):

        audio_path = Path(
            SESSION_ROW[s][
                f"p{p}_audio_path"
            ]
        )

        if not audio_path.exists():
            continue

        off, corr = estimate_audio_offset(
            s,
            p,
        )

        warning = bool(
            abs(off)
            > float(
                ALIGNMENT_WARN_ABS_S
            )
            or abs(
                abs(off)
                - float(
                    ALIGNMENT_SEARCH_MAX_S
                )
            )
            <= float(
                ALIGNMENT_BOUNDARY_MARGIN_S
            )
        )

        audio_audit.append(
            {
                "session": s,
                "participant": p,
                "offset_s": float(off),
                "corr": float(corr),
                "warning": warning,
            }
        )

AUDIO_ALIGNMENT_AUDIT = pd.DataFrame(
    audio_audit
)
AUDIO_ALIGNMENT_FINGERPRINT=_support_table_fingerprint(AUDIO_ALIGNMENT_AUDIT,["session","participant","offset_s","corr","warning"])
print("Audio-alignment fingerprint:",AUDIO_ALIGNMENT_FINGERPRINT)

display(
    AUDIO_ALIGNMENT_AUDIT
)

if (
    len(AUDIO_ALIGNMENT_AUDIT)
    and AUDIO_ALIGNMENT_AUDIT[
        "warning"
    ].any()
):

    print(
        "WARNING: some audio offsets are outside the "
        "expected small-offset regime. "
        "No affine scaling is ever applied."
    )

ACOUSTIC_CACHE_DIR = (
    DETERMINISTIC_CACHE_ROOT
    / f"cheap_audio_10hz_{AUDIO_ALIGNMENT_FINGERPRINT}"
)

ACOUSTIC_CACHE_DIR.mkdir(
    exist_ok=True,
    parents=True,
)

def build_cheap_audio_cache(
    session,
):
    session = str(session)

    path = (
        ACOUSTIC_CACHE_DIR
        / f"{session}.npz"
    )

    if path.exists():
        try:
            _old=np.load(path,allow_pickle=False)
            _ok=(
                "data_fingerprint" in _old.files and str(_old["data_fingerprint"].item())==str(MULTITPC_DATA_FINGERPRINT)
                and "audio_alignment_fingerprint" in _old.files and str(_old["audio_alignment_fingerprint"].item())==str(AUDIO_ALIGNMENT_FINGERPRINT)
                and "alignment_contract_rev" in _old.files and str(_old["alignment_contract_rev"].item())==str(ALIGNMENT_CONTRACT_REV)
            )
            _old.close()
            if _ok:return path
        except Exception:pass
        path.unlink(missing_ok=True)

    ann = load_annotation(
        session
    )

    dur = (
        len(ann["audio"])
        / float(FRAME_HZ)
    )

    times = np.arange(
        0.05,
        dur,
        0.10,
        dtype=np.float32,
    )

    feats = np.zeros(
        (
            len(times),
            3,
            3,
        ),
        dtype=np.float32,
    )

    for pi, p in enumerate(
        (1, 2, 3)
    ):

        x, sr = read_participant_audio(
            session,
            p,
        )

        off, _ = estimate_audio_offset(
            session,
            p,
        )

        for i, t in enumerate(times):

            raw_t = (
                float(t)
                + float(off)
            )

            hi = int(
                round(
                    raw_t
                    * float(sr)
                )
            )

            lo = int(
                round(
                    (
                        raw_t
                        - 0.10
                    )
                    * float(sr)
                )
            )

            lo = max(
                0,
                lo,
            )

            hi = min(
                len(x),
                hi,
            )

            if hi <= lo:
                continue

            seg = x[lo:hi]

            if len(seg) == 0:
                continue

            feats[
                i,
                pi,
                0,
            ] = np.sqrt(
                np.mean(
                    seg * seg
                )
                + 1e-10
            )

            if len(seg) > 1:
                feats[
                    i,
                    pi,
                    1,
                ] = np.mean(
                    np.abs(
                        np.diff(seg)
                    )
                )

            feats[
                i,
                pi,
                2,
            ] = np.max(
                np.abs(seg)
            )

    np.savez_compressed(
        path,
        times=times,
        features=feats,
        alignment_method=np.asarray(
            "verified_offset_only"
        ),
        scale=np.float32(1.0),
        frame_hz=np.float32(10.0),
        data_fingerprint=np.asarray(MULTITPC_DATA_FINGERPRINT),
        audio_alignment_fingerprint=np.asarray(AUDIO_ALIGNMENT_FINGERPRINT),
        alignment_contract_rev=np.asarray(ALIGNMENT_CONTRACT_REV),
    )

    return path

built = 0
skipped = 0

for s in MANIFEST[
    "session"
].astype(str):

    have_all_audio = all(
        Path(
            SESSION_ROW[s][
                f"p{p}_audio_path"
            ]
        ).exists()
        for p in (1, 2, 3)
    )

    if not have_all_audio:
        skipped += 1
        print(
            f"WARNING: skipping cheap audio cache for {s}: "
            f"one or more participant WAV files are missing."
        )
        continue

    build_cheap_audio_cache(
        s
    )

    built += 1

print(
    f"Cheap acoustic caches ready: "
    f"{built} sessions built/reused, "
    f"{skipped} skipped."
)

Audio-alignment fingerprint: 80788b5cfc1b0e4fa95e39aa


,session,participant,offset_s,corr,warning
0,mtpc_20220128_s01,1,0.150000,0.412716,False
1,mtpc_20220128_s01,2,0.150000,0.418396,False
2,mtpc_20220128_s01,3,0.150000,0.444368,False
3,mtpc_20220128_s02,1,0.150000,0.456755,False
4,mtpc_20220128_s02,2,0.183333,0.294372,False
...,...,...,...,...,...
64,mtpc_20211215_s03,2,0.150000,0.590542,False
65,mtpc_20211215_s03,3,0.133333,0.583576,False
66,mtpc_20211215_s04,1,0.150000,0.398898,False
67,mtpc_20211215_s04,2,0.166667,0.554771,False


Cheap acoustic caches ready: 23 sessions built/reused, 0 skipped.


In [6]:
# Prosody and motion loaders
MODALITY_ALIGNMENT_AUDIT=[]

def _safe_row_align_numeric(num,n_native,session,modality,participant=None):
    num=np.asarray(num,np.float32)
    if num.ndim==1:num=num[:,None]
    mismatch_frames=int(len(num)-int(n_native))
    mismatch_s=float(mismatch_frames/FRAME_HZ)
    safe=abs(mismatch_s)<=MODALITY_ROW_MISMATCH_MAX_S
    inactive_tail_exception=False
    if (not safe) and len(num)<int(n_native):
        ann_audio=load_annotation(str(session))["audio"]
        tail=ann_audio[len(num):int(n_native)]
        inactive_tail_exception=bool(len(tail) and np.all(tail==0))
        safe=inactive_tail_exception
    out=np.full((int(n_native),num.shape[1]),np.nan,np.float32)
    valid=np.zeros(int(n_native),bool)
    if safe:
        m=min(int(n_native),len(num))
        if m>0:
            out[:m]=num[:m]
            valid[:m]=np.isfinite(out[:m]).any(axis=1)
        action="safe_shorter_inactive_native_tail" if inactive_tail_exception else "row_aligned_with_small_tail_pad_or_trim"
    else:
        action="invalidated_large_length_mismatch"
    MODALITY_ALIGNMENT_AUDIT.append(dict(
        session=str(session),modality=str(modality),participant=participant,
        native_frames=int(n_native),source_frames=int(len(num)),
        mismatch_frames=mismatch_frames,mismatch_s=mismatch_s,safe=bool(safe),action=action
    ))
    return out,valid

PROSODY_COLUMN_AUDIT=[]

def _norm_colname(x):
    return re.sub(r"[^a-z0-9]+","_",str(x).strip().lower()).strip("_")

@lru_cache(maxsize=64)
def load_prosody(session):
    path=Path(SESSION_ROW[str(session)]["prosody_path"]);n=len(load_annotation(session)["audio"])
    if not path.exists():
        MODALITY_ALIGNMENT_AUDIT.append(dict(session=str(session),modality="prosody",participant=None,native_frames=n,source_frames=0,
                                             mismatch_frames=-n,mismatch_s=-n/FRAME_HZ,safe=False,action="missing"))
        PROSODY_COLUMN_AUDIT.append(dict(session=str(session),mode="missing",columns=[]))
        return np.full((n,6),np.nan,np.float32),np.zeros(n,bool)

    df=pd.read_csv(path)
    normalized={_norm_colname(c):c for c in df.columns}
    wanted=["p1_pitch","p1_intensity","p2_pitch","p2_intensity","p3_pitch","p3_intensity"]
    if PROSODY_COLUMN_OVERRIDE is not None:
        selected=list(PROSODY_COLUMN_OVERRIDE)
        if len(selected)!=6 or any(c not in df.columns for c in selected):
            raise RuntimeError(f"{session}: invalid PROSODY_COLUMN_OVERRIDE={selected}; observed={list(df.columns)}")
        mode="explicit_override"
    elif all(k in normalized for k in wanted):
        selected=[normalized[k] for k in wanted];mode="named_pitch_intensity"
    else:
        raise RuntimeError(
            f"{session}: canonical p1/p2/p3 pitch+intensity columns were not found. "
            f"Observed columns={list(df.columns)}. Set PROSODY_COLUMN_OVERRIDE explicitly for a renamed release."
        )
    num=df[selected].apply(pd.to_numeric,errors="coerce").to_numpy(np.float32)
    PROSODY_COLUMN_AUDIT.append(dict(session=str(session),mode=mode,columns=[str(x) for x in selected]))
    return _safe_row_align_numeric(num,n,session,"prosody",None)

@lru_cache(maxsize=128)
def load_mocap(session,p):
    path=Path(SESSION_ROW[str(session)][f"p{int(p)}_mocap_path"]);n=len(load_annotation(session)["audio"])
    if not path.exists():
        MODALITY_ALIGNMENT_AUDIT.append(dict(session=str(session),modality="mocap",participant=int(p),native_frames=n,source_frames=0,
                                             mismatch_frames=-n,mismatch_s=-n/FRAME_HZ,safe=False,action="missing"))
        return np.full((n,0),np.nan,np.float32),np.zeros(n,bool)
    try:df=pd.read_csv(path,sep=None,engine="python",header=None)
    except Exception:df=pd.read_csv(path,sep=r"\s+",header=None)
    num=df.apply(pd.to_numeric,errors="coerce").to_numpy(np.float32)
    return _safe_row_align_numeric(num,n,session,"mocap",int(p))

VALIDITY=[]
for s in MANIFEST.session:
    n=len(load_annotation(s)["audio"])
    pr,pv=load_prosody(s);mvals=[]
    for p in (1,2,3):
        m,mv=load_mocap(s,p);mvals.append(float(mv.mean()) if len(mv) else 0.0)
    VALIDITY.append(dict(session=s,native_frames=n,prosody_valid=float(pv.mean()) if len(pv) else 0.0,mocap_valid_mean=float(np.mean(mvals))))
VALIDITY_DF=pd.DataFrame(VALIDITY)
MODALITY_ALIGNMENT_AUDIT_DF=pd.DataFrame(MODALITY_ALIGNMENT_AUDIT).drop_duplicates(
    ["session","modality","participant"],keep="last"
).reset_index(drop=True)
PROSODY_COLUMN_AUDIT_DF=pd.DataFrame(PROSODY_COLUMN_AUDIT).drop_duplicates("session",keep="last").reset_index(drop=True)
display(VALIDITY_DF)
display(MODALITY_ALIGNMENT_AUDIT_DF.sort_values(["safe","session","modality"]))
display(PROSODY_COLUMN_AUDIT_DF)

,session,native_frames,prosody_valid,mocap_valid_mean
0,mtpc_20220128_s01,19377,0.998607,1.0
1,mtpc_20220128_s02,33463,0.999253,1.0
2,mtpc_20220128_s03,21566,0.656589,1.0
3,mtpc_20220128_s04,36625,0.999317,1.0
4,mtpc_20220128_s05,25730,0.834823,1.0
5,mtpc_20220128_s06,19681,0.998730,1.0
6,mtpc_20220211_s02,43968,0.999454,0.0
7,mtpc_20220211_s03,68872,0.999593,1.0
8,mtpc_20220304_s01,47308,0.999408,1.0
9,mtpc_20220304_s02,49953,0.999459,1.0


,session,modality,participant,native_frames,source_frames,mismatch_frames,mismatch_s,safe,action
25,mtpc_20220211_s02,mocap,1.0,43968,55534,11566,192.766667,False,invalidated_large_length_mismatch
26,mtpc_20220211_s02,mocap,2.0,43968,55534,11566,192.766667,False,invalidated_large_length_mismatch
27,mtpc_20220211_s02,mocap,3.0,43968,55534,11566,192.766667,False,invalidated_large_length_mismatch
77,mtpc_20211215_s01,mocap,1.0,36298,36298,0,0.000000,True,row_aligned_with_small_tail_pad_or_trim
78,mtpc_20211215_s01,mocap,2.0,36298,36298,0,0.000000,True,row_aligned_with_small_tail_pad_or_trim
...,...,...,...,...,...,...,...,...,...
68,mtpc_20220312_s02,prosody,NaN,52719,52692,-27,-0.450000,True,row_aligned_with_small_tail_pad_or_trim
73,mtpc_20220312_s03,mocap,1.0,60370,60370,0,0.000000,True,row_aligned_with_small_tail_pad_or_trim
74,mtpc_20220312_s03,mocap,2.0,60370,60370,0,0.000000,True,row_aligned_with_small_tail_pad_or_trim
75,mtpc_20220312_s03,mocap,3.0,60370,60370,0,0.000000,True,row_aligned_with_small_tail_pad_or_trim


,session,mode,columns
0,mtpc_20220128_s01,named_pitch_intensity,"[p1_pitch, p1_intensity, p2_pitch, p2_intensit..."
1,mtpc_20220128_s02,named_pitch_intensity,"[p1_pitch, p1_intensity, p2_pitch, p2_intensit..."
2,mtpc_20220128_s03,named_pitch_intensity,"[p1_pitch, p1_intensity, p2_pitch, p2_intensit..."
3,mtpc_20220128_s04,named_pitch_intensity,"[p1_pitch, p1_intensity, p2_pitch, p2_intensit..."
4,mtpc_20220128_s05,named_pitch_intensity,"[p1_pitch, p1_intensity, p2_pitch, p2_intensit..."
5,mtpc_20220128_s06,named_pitch_intensity,"[p1_pitch, p1_intensity, p2_pitch, p2_intensit..."
6,mtpc_20220211_s02,named_pitch_intensity,"[p1_pitch, p1_intensity, p2_pitch, p2_intensit..."
7,mtpc_20220211_s03,named_pitch_intensity,"[p1_pitch, p1_intensity, p2_pitch, p2_intensit..."
8,mtpc_20220304_s01,named_pitch_intensity,"[p1_pitch, p1_intensity, p2_pitch, p2_intensit..."
9,mtpc_20220304_s02,named_pitch_intensity,"[p1_pitch, p1_intensity, p2_pitch, p2_intensit..."


In [7]:
# Native state mappings
LEFT_RIGHT_TO_RECIPIENT = {1: {1: 2, 2: 3}, 2: {1: 3, 2: 1}, 3: {1: 1, 2: 2}}

def paper_isv(native):
    x = np.asarray(native, np.int16)
    y = np.zeros_like(x)
    y[x == 1] = 2
    y[x == 2] = 1
    return y

def paper_gtv(relative):
    x = np.asarray(relative, np.int16)
    y = np.zeros_like(x)
    for pi, p in enumerate((1, 2, 3)):
        y[:, pi] = np.where(x[:, pi] == 0, 0, np.where(x[:, pi] == 1, LEFT_RIGHT_TO_RECIPIENT[p][1], LEFT_RIGHT_TO_RECIPIENT[p][2]))
    return y

In [8]:
# Native NTT/ITT event definitions
def runs(mask):
    x=np.asarray(mask,bool)
    d=np.diff(np.r_[False,x,False].astype(np.int8))
    return list(zip(np.where(d==1)[0],np.where(d==-1)[0]))

def _primary_segments(session):
    a=load_annotation(session)["audio"]
    seg=[]
    for p in range(3):
        for st,en in runs(a[:,p]==1):
            if en-st>=2:
                seg.append((int(st),int(en),int(p)))
    return sorted(seg,key=lambda z:(z[0],z[1],z[2]))

def _support_resolve_same_transfer_frame_candidates(rows):
    """
    Collapse duplicate transfer interpretations caused only by >1 incumbent.

    A duplicate is collapsible iff native frame, event subtype, and recipient
    are identical. The transfer itself is therefore unambiguous; only `prev`
    is ambiguous. We keep one deterministic representative `prev` for table
    compatibility and expose the ambiguity explicitly:

      * ITT: prefer the incumbent that releases latest after the incoming onset
        (the last incumbent to yield the floor);
      * ties / NTT: prefer the most recently started incumbent segment;
      * final deterministic tie-break: lower participant id.

    The representative is NOT treated as oracle truth in previous-holder-
    dependent next-speaker evaluation when prev_ambiguous=True.
    """
    cols=[
        "session","time","event","prev","next",
        "floor_release_time","floor_acquisition_time",
        "transition_interval_start","transition_interval_end",
        "prev_ambiguous","prev_candidates",
    ]
    if not rows:
        return pd.DataFrame(columns=cols)

    q=pd.DataFrame(rows).copy()
    q["native_frame"]=np.rint(
        pd.to_numeric(q["time"],errors="raise").to_numpy(float)*FRAME_HZ
    ).astype(np.int64)

    resolved=[]
    group_cols=["session","native_frame","event","next"]
    for (_,native_frame,_,_),g in q.groupby(group_cols,sort=True,dropna=False):
        g=g.copy()
        prevs=tuple(sorted({int(x) for x in g["prev"].tolist()}))

        g=g.sort_values(
            ["floor_release_time","_prev_start_time","prev"],
            ascending=[False,False,True],
            kind="mergesort",
        )
        r=g.iloc[0].copy()
        r["time"]=float(int(native_frame)/FRAME_HZ)
        r["prev_ambiguous"]=bool(len(prevs)>1)
        r["prev_candidates"]="|".join(map(str,prevs))
        resolved.append(r)

    out=pd.DataFrame(resolved)

    dup=out.duplicated(["session","native_frame"],keep=False)
    if dup.any():
        bad=out.loc[
            dup,
            ["session","native_frame","time","event","prev","next","prev_ambiguous","prev_candidates"],
        ].sort_values(["session","native_frame","event","next","prev"])
        raise RuntimeError(
            "Conflicting simultaneous research transfer events remain after "
            "same-event/same-recipient incumbent collapse. Resolve the event "
            "ontology before training:\n"+bad.to_string(index=False)
        )

    out=out.drop(columns=["native_frame","_prev_start_time"],errors="ignore")
    return (
        out[cols]
        .sort_values(["time","event","next","prev"])
        .reset_index(drop=True)
    )

@lru_cache(maxsize=64)
def derive_floor_events(session):
    """
    Frozen research event contract.

    NTT:
      current primary speaker releases;
      chronologically next primary run is another participant;
      gap >= 200 ms;
      timestamp = outgoing release.

    ITT:
      another primary speaker starts before current speaker releases;
      outgoing speaker resolves within ITT_RESOLUTION_S;
      incoming speaker maintains primary speech for at least
      ITT_MIN_SUCCESSFUL_FLOOR_S after outgoing release;
      timestamp = incoming onset.

    Triadic ambiguity:
      if multiple incumbents generate the same transfer frame/subtype/recipient,
      emit ONE event with prev_ambiguous=True and prev_candidates recorded.
    """
    session=str(session)
    seg=_primary_segments(session)
    rr=[]

    for st,en,p in seg:
        inter=[]
        for qst,qen,qp in seg:
            if qp==p or not (st<qst<en):
                continue
            resolution=(en-qst)/FRAME_HZ
            successful_after=(qen-en)/FRAME_HZ
            if (
                resolution<=ITT_RESOLUTION_S+1e-9
                and successful_after>=ITT_MIN_SUCCESSFUL_FLOOR_S-1e-9
            ):
                inter.append((qst,qen,qp))

        if inter:
            qst,qen,qp=min(inter,key=lambda z:(z[0],z[1],z[2]))
            rr.append(dict(
                session=session,
                time=float(qst/FRAME_HZ),
                event="interruption",
                prev=int(p+1),
                next=int(qp+1),
                floor_release_time=float(en/FRAME_HZ),
                floor_acquisition_time=float(qst/FRAME_HZ),
                transition_interval_start=float(qst/FRAME_HZ),
                transition_interval_end=float(en/FRAME_HZ),
                _prev_start_time=float(st/FRAME_HZ),
            ))
            continue

        after=[q for q in seg if q[0]>=en]
        if after:
            qst,qen,qp=min(after,key=lambda z:(z[0],z[1],z[2]))
            gap=(qst-en)/FRAME_HZ
            if qp!=p and gap>=NATIVE_NTT_MIN_GAP_S-1e-9:
                rr.append(dict(
                    session=session,
                    time=float(en/FRAME_HZ),
                    event="turn_taking",
                    prev=int(p+1),
                    next=int(qp+1),
                    floor_release_time=float(en/FRAME_HZ),
                    floor_acquisition_time=float(qst/FRAME_HZ),
                    transition_interval_start=float(en/FRAME_HZ),
                    transition_interval_end=float(qst/FRAME_HZ),
                    _prev_start_time=float(st/FRAME_HZ),
                ))

    return _support_resolve_same_transfer_frame_candidates(rr)

@lru_cache(maxsize=64)
def derive_behavior_diagnostics(session):
    """
    Native-state diagnostic events used ONLY to label structured negatives.
    They are never promoted to transfer positives.
    """
    session=str(session)
    a=load_annotation(session)["audio"]
    seg=_primary_segments(session)
    rows=[]

    for p in range(3):
        for st,en in runs(a[:,p]==2):
            if en-st>=1:
                rows.append(dict(
                    session=session,time=st/FRAME_HZ,
                    event="backchannel",participant=p+1
                ))

    true_itt=set(
        (int(round(r.time*FRAME_HZ)),int(r.next))
        for r in derive_floor_events(session).itertuples(index=False)
        if str(r.event)=="interruption"
    )
    for st,en,p in seg:
        for qst,qen,qp in seg:
            if qp==p or not (st<qst<en):
                continue
            key=(int(qst),int(qp+1))
            if key not in true_itt:
                rows.append(dict(
                    session=session,time=qst/FRAME_HZ,
                    event="nontransfer_overlap",participant=qp+1
                ))

    byp={p:[] for p in range(3)}
    for st,en,p in seg:
        byp[p].append((st,en))
    for p,ss in byp.items():
        ss=sorted(ss)
        for (st,en),(nst,nen) in zip(ss[:-1],ss[1:]):
            gap=(nst-en)/FRAME_HZ
            if gap>=NATIVE_NTT_MIN_GAP_S:
                intervening=[x for x in seg if en<=x[0]<nst and x[2]!=p]
                if not intervening:
                    rows.append(dict(
                        session=session,time=en/FRAME_HZ,
                        event="same_speaker_continuation",participant=p+1
                    ))

    return (
        pd.DataFrame(rows)
        if rows else
        pd.DataFrame(columns=["session","time","event","participant"])
    )

def _support_assert_unique_transfer_events(events):
    """Fail only if >1 transfer interpretation still remains at one native frame."""
    if events is None or not len(events):
        return True
    q=events.copy()
    q["native_frame"]=np.rint(
        pd.to_numeric(q["time"],errors="raise").to_numpy(float)*FRAME_HZ
    ).astype(np.int64)
    dup=q.duplicated(["session","native_frame"],keep=False)
    if dup.any():
        cols=["session","native_frame","time","event","prev","next"]
        for extra in ["prev_ambiguous","prev_candidates"]:
            if extra in q.columns:
                cols.append(extra)
        bad=q.loc[dup,cols].sort_values(["session","native_frame","event","next","prev"])
        raise RuntimeError(
            "Conflicting simultaneous research transfer events remain at the same "
            "native 60-Hz frame after ambiguity resolution:\n"+bad.to_string(index=False)
        )
    return True

EVENT_DIAGNOSTIC_SESSIONS = list(map(str, MANIFEST['session']))
EVENTS_ALL = pd.concat([derive_floor_events(s) for s in EVENT_DIAGNOSTIC_SESSIONS], ignore_index=True)
_support_assert_unique_transfer_events(EVENTS_ALL)
BEHAVIOR_DIAGNOSTICS = pd.concat([derive_behavior_diagnostics(s) for s in EVENT_DIAGNOSTIC_SESSIONS], ignore_index=True)
if set(EVENTS_ALL['session'].astype(str)) != set(EVENT_DIAGNOSTIC_SESSIONS):
    missing = sorted(set(EVENT_DIAGNOSTIC_SESSIONS) - set(EVENTS_ALL['session'].astype(str)))
    if missing:
        print('Sessions without transfer events:', missing)
print('Post-hoc event table:', len(EVENTS_ALL), 'events across', EVENTS_ALL['session'].nunique(), 'sessions')

Post-hoc event table: 2511 events across 23 sessions


In [9]:
# Deterministic causal 10-Hz feature table; labels are stored separately from X
CAUSAL_FEATURE_CACHE = DEFAULT_CAUSAL_CACHE
CAUSAL_FEATURE_CACHE.mkdir(parents=True, exist_ok=True)

def _causal_roll(x, width, reducer):
    frame = pd.DataFrame(np.asarray(x, np.float32))
    roll = frame.rolling(int(width), min_periods=1)
    if reducer == 'mean':
        return roll.mean().to_numpy(np.float32)
    if reducer == 'std':
        return roll.std(ddof=0).fillna(0).to_numpy(np.float32)
    raise ValueError(reducer)

def _nearest_event_payload(session, times):
    ev = EVENTS_ALL[EVENTS_ALL['session'].astype(str).eq(str(session))].sort_values('time').reset_index(drop=True)
    q = np.asarray(times, float)
    if ev.empty:
        return {
            'distance': np.full(len(q), np.inf), 'event_time': np.full(len(q), np.nan),
            'event_type': np.full(len(q), -1), 'next_speaker': np.full(len(q), -1),
        }
    et = ev['time'].to_numpy(float)
    right = np.clip(np.searchsorted(et, q, side='left'), 0, len(et) - 1)
    left = np.clip(right - 1, 0, len(et) - 1)
    choose_left = np.abs(q - et[left]) <= np.abs(q - et[right])
    idx = np.where(choose_left, left, right)
    return {
        'distance': np.abs(q - et[idx]),
        'event_time': et[idx],
        'event_type': np.where(ev['event'].astype(str).to_numpy()[idx] == 'turn_taking', 0, 1),
        'next_speaker': ev['next'].to_numpy(int)[idx],
    }

def build_session_feature_table(session):
    session = str(session)
    cache = CAUSAL_FEATURE_CACHE / f'{session}.pkl'
    contract = _support_sha256_json({'protocol': BASE_PROTOCOL_ID, 'implementation': IMPLEMENTATION_SHA256, 'feature_source': inspect.getsource(build_session_feature_table), 'label_source': inspect.getsource(_nearest_event_payload), 'label_tolerance': PRIMARY_EVENT_TOLERANCE, 'data': MULTITPC_DATA_FINGERPRINT, 'alignment': AUDIO_ALIGNMENT_FINGERPRINT}, n=24)
    if cache.is_file():
        old = pd.read_pickle(cache)
        if old.attrs.get('contract') == contract:
            return old

    audio_path = build_cheap_audio_cache(session)
    with np.load(audio_path, allow_pickle=False) as d:
        times = np.asarray(d['times'], np.float32)
        raw = np.asarray(d['features'], np.float32).reshape(len(times), -1)
    log_audio = np.sign(raw) * np.log1p(np.abs(raw) * 100.0)
    blocks = [log_audio]
    names = [f'audio_now_{i}' for i in range(log_audio.shape[1])]
    for lag_s in (0.2, 0.5, 1.0):
        lag = int(round(lag_s / 0.1))
        prev = log_audio[np.maximum(np.arange(len(log_audio))-lag,0)]
        blocks.append(log_audio - prev)
        names += [f'audio_delta_{lag_s:g}s_{i}' for i in range(log_audio.shape[1])]
    for win_s in (0.5, 1.0, 2.0):
        width = int(round(win_s / 0.1))
        blocks += [_causal_roll(log_audio, width, 'mean'), _causal_roll(log_audio, width, 'std')]
        names += [f'audio_mean_{win_s:g}s_{i}' for i in range(log_audio.shape[1])]
        names += [f'audio_std_{win_s:g}s_{i}' for i in range(log_audio.shape[1])]

    ann = load_annotation(session)
    idx = np.clip(np.floor(times * FRAME_HZ + 1e-9).astype(int), 0, len(ann['audio']) - 1)
    gaze = paper_gtv(np.asarray(ann['gaze'], np.int16)[idx]).astype(np.float32) / 3.0
    blocks.append(gaze)
    names += [f'gaze_p{i}' for i in (1, 2, 3)]

    prosody, prosody_valid = load_prosody(session)
    pros = np.nan_to_num(np.asarray(prosody, np.float32)[idx])
    blocks += [pros, np.asarray(prosody_valid, np.float32)[idx, None]]
    names += [f'prosody_{i}' for i in range(pros.shape[1])] + ['prosody_valid']

    motion_parts = []
    for participant in (1, 2, 3):
        motion, valid = load_mocap(session, participant)
        values = np.zeros((len(times), 4), np.float32)
        if motion.ndim == 2 and motion.shape[1]:
            now = np.asarray(motion, np.float32)[idx]
            prev_idx = np.clip(idx - int(round(0.2 * FRAME_HZ)), 0, len(motion) - 1)
            diff = np.nan_to_num(np.abs(now - np.asarray(motion, np.float32)[prev_idx]),nan=0.,posinf=0.,neginf=0.)
            values[:, 0] = np.nanmean(diff, axis=1)
            values[:, 1] = np.nanstd(diff, axis=1)
            values[:, 2] = np.nanmax(diff, axis=1)
            values[:, 3] = np.asarray(valid, np.float32)[idx]
        motion_parts.append(np.nan_to_num(values))
    motion = np.concatenate(motion_parts, axis=1)
    blocks.append(motion)
    names += [f'motion_{i}' for i in range(motion.shape[1])]

    X = np.concatenate(blocks, axis=1).astype(np.float32)
    payload = _nearest_event_payload(session, times)
    table = pd.DataFrame({'session': session, 'time': times.astype(float)})
    for i, name in enumerate(names):
        table[name] = X[:, i]
    table['nearest_event_distance'] = payload['distance']
    table['event_time'] = payload['event_time']
    table['label'] = (payload['distance'] <= PRIMARY_EVENT_TOLERANCE + 1e-9).astype(np.int8)
    table['event_type'] = np.where(table['label'].to_numpy(bool), payload['event_type'], -1).astype(np.int8)
    table['next_speaker'] = np.where(table['label'].to_numpy(bool), payload['next_speaker'], -1).astype(np.int8)
    table['target_offset'] = np.where(table['label'].to_numpy(bool), payload['event_time'] - times, 0.0).astype(np.float32)
    table.attrs['contract'] = contract
    table.attrs['feature_names'] = names
    table.to_pickle(cache)
    return table

SESSION_TABLES = {s: build_session_feature_table(s) for s in MANIFEST['session'].astype(str)}
BASE_FEATURE_NAMES = list(next(iter(SESSION_TABLES.values())).attrs['feature_names'])
if any(list(t.attrs['feature_names']) != BASE_FEATURE_NAMES for t in SESSION_TABLES.values()):
    raise RuntimeError('Causal feature contract drift across sessions.')
print('Causal features:', len(BASE_FEATURE_NAMES), 'dimensions')

  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table['nearest_event_distance'] = payload['distance']
  table['event_time'] = payload['event_time']
  table['label'] = (payload['distance'] <= PRIMARY_EVENT_TOLERANCE + 1e-9).astype(np.int8)
  table['event_type'] = np.where(table['label'].to_numpy(bool), payload['event_type'], -1).astype(np.int8)
  table['next_speaker'] = np.where(table['label'].to_numpy(bool), payload['next_speaker'], -1).astype(np.int8)
  table['target_offset'] = np.where(table['label'].to_numpy(bool), payload['event_time'] - times, 0.0).astype(np.float32)
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[nam

Causal features: 112 dimensions


  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table['nearest_event_distance'] = payload['distance']
  table['event_time'] = payload['event_time']
  table['label'] = (payload['distance'] <= PRIMARY_EVENT_TOLERANCE + 1e-9).astype(np.int8)
  table['event_type'] = np.where(table['label'].to_numpy(bool), payload['event_type'], -1).astype(np.int8)
  table['next_speaker'] = np.where(table['label'].to_numpy(bool), payload['next_speaker'], -1).astype(np.int8)
  table['target_offset'] = np.where(table['label'].to_numpy(bool), payload['event_time'] - times, 0.0).astype(np.float32)
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[name] = X[:, i]
  table[nam

In [10]:
# Raw aligned participant-audio history reader
def _raw_audio_interval(session, p, start_t, end_t):
    start_t = float(start_t)
    end_t = float(end_t)
    out_n = max(1, int(round(max(0.0, end_t - start_t) * TARGET_SR)))
    out = np.zeros(out_n, np.float32)
    if end_t <= start_t:
        return out[:0]
    path = Path(SESSION_ROW[str(session)][f'p{int(p)}_audio_path'])
    off, _ = estimate_audio_offset(session, p)
    raw0 = start_t + float(off)
    raw1 = end_t + float(off)
    info = sf.info(str(path))
    sr = int(info.samplerate)
    dur = float(info.frames) / sr
    clip0 = max(0.0, raw0)
    clip1 = min(dur, raw1)
    if clip1 <= clip0:
        return out
    s0 = max(0, int(math.floor(clip0 * sr)))
    s1 = min(int(info.frames), int(math.ceil(clip1 * sr)))
    x, _ = sf.read(str(path), start=s0, frames=max(0, s1 - s0), dtype='float32', always_2d=False)
    x = np.asarray(x, np.float32)
    if x.ndim > 1:
        x = x.mean(1)
    x = np.nan_to_num(x).astype(np.float32)
    if sr != TARGET_SR:
        x = resample_linear(x, sr, TARGET_SR)
    dst = int(round((s0 / sr - raw0) * TARGET_SR))
    if dst < 0:
        x = x[-dst:]
        dst = 0
    if dst < out_n and len(x):
        take = min(len(x), out_n - dst)
        out[dst:dst + take] = x[:take]
    return out

In [11]:
# Causal WavLM/ECAPA feature adapter
def _read_mixed_audio_history(session, t, context_s):
    waves = [_raw_audio_interval(str(session), p, float(t) - float(context_s), float(t)) for p in (1, 2, 3)]
    n = max([len(w) for w in waves] + [320])
    out = []
    for w in waves:
        w = np.asarray(w, np.float32)
        if len(w) < n:
            w = np.pad(w, (n - len(w), 0))
        elif len(w) > n:
            w = w[-n:]
        peak = float(np.quantile(np.abs(w), 0.995)) if len(w) else 0.0
        if peak > 1e-06:
            w = w / max(1.0, peak)
        out.append(w)
    return np.mean(np.stack(out, axis=0), axis=0).astype(np.float32)

def _l2_normalize(x):
    if x is None:
        return None
    x = np.asarray(x, np.float32).reshape(-1)
    if not np.isfinite(x).all():
        return None
    n = float(np.linalg.norm(x))
    if n <= 1e-08:
        return None
    return (x / n).astype(np.float32)

def _cosine_with_validity(a, b):
    a = _l2_normalize(a)
    b = _l2_normalize(b)
    if a is None or b is None:
        return (0.0, 0.0)
    return (float(np.clip(np.dot(a, b), -1.0, 1.0)), 1.0)

def _patch_huggingface_hub_compat():
    try:
        import huggingface_hub
        original = huggingface_hub.hf_hub_download
        if getattr(original, '_r11_auth_patch', False):
            return True
        try:
            params = set(inspect.signature(original).parameters)
        except Exception:
            params = set()
        if 'use_auth_token' in params:
            return True

        def compat(*args, use_auth_token=None, token=None, **kwargs):
            if token is None and use_auth_token not in [None, False]:
                token = use_auth_token
            return original(*args, token=token, **kwargs)
        compat._hf_auth_patch = True
        huggingface_hub.hf_hub_download = compat
        for modname in ('speechbrain.utils.fetching', 'speechbrain.pretrained.fetching'):
            try:
                mod = __import__(modname, fromlist=['hf_hub_download'])
                setattr(mod, 'hf_hub_download', compat)
            except Exception:
                pass
        return True
    except Exception:
        return False

def _resolve_ecapa_classifier():
    errors = []
    for path in ('speechbrain.inference.speaker', 'speechbrain.inference.classifiers', 'speechbrain.pretrained'):
        try:
            mod = __import__(path, fromlist=['EncoderClassifier'])
            return (getattr(mod, 'EncoderClassifier'), path)
        except Exception as e:
            errors.append((path, repr(e)))
    raise ImportError(f'Could not import SpeechBrain EncoderClassifier: {errors}')

class CausalAudioFeatureAdapter:
    def __init__(self):
        self.available=False
        self.reason=""
        self.wavlm=None
        self.ecapa=None
        self.device=DEVICE
        self.model_name=WAVLM_MODEL_ID
        self.ecapa_source=ECAPA_MODEL_ID
        self.wavlm_commit="unknown"
        self.wavlm_config_fingerprint="unknown"
        self.ecapa_cache_fingerprint="unknown"
        self.cache_contract={}
        self.fp="uninitialized"
        self.cache_dir=None
        # Initialise before _wavlm_features() reads it.
        self._wavlm_projection=None
        try:
            from transformers import WavLMModel
            if self.device.type=="cuda" and not ensure_min_cuda_free("causal WavLM-large/ECAPA adapter",4.0):
                self.device=torch.device("cpu")
            self.wavlm=WavLMModel.from_pretrained(self.model_name,revision=WAVLM_MODEL_REVISION)
            self.wavlm.to(self.device).eval()
            for p in self.wavlm.parameters():p.requires_grad=False
            self.wavlm_commit=str(getattr(self.wavlm.config,"_commit_hash",None) or "unknown")

            _patch_huggingface_hub_compat()
            EC,import_path=_resolve_ecapa_classifier()
            savedir=AUDIO_FEATURE_CACHE_ROOT/"speechbrain_ecapa_voxceleb"
            savedir.mkdir(parents=True,exist_ok=True)
            kwargs=dict(source=self.ecapa_source,savedir=str(savedir),run_opts={"device":str(self.device)})
            # Use the standard SpeechBrain call; passing pymodule_file=None is version-sensitive.
            self.ecapa=EC.from_hparams(**kwargs)
            self.ecapa_import_path=import_path

            # Bind cached features to the loaded pretrained resources.
            self.wavlm_config_fingerprint=_support_sha256_json(self.wavlm.config.to_dict(),n=24)
            _ecapa_files=sorted(
                [q for q in savedir.rglob("*") if q.is_file()],
                key=lambda q:str(q)
            )
            _ecapa_identity=[]
            for q in _ecapa_files:
                rel=str(q.relative_to(savedir))
                try:
                    rec=_support_file_identity(q)
                    _ecapa_identity.append(dict(relative_path=rel,sha256=str(rec["sha256"])))
                except Exception:
                    st=q.stat()
                    _ecapa_identity.append(dict(relative_path=rel,size=int(st.st_size)))
            self.ecapa_cache_fingerprint=_support_sha256_json(_ecapa_identity,n=24)
            try:_support_flush_file_hash_index()
            except Exception:pass

            self.cache_contract=dict(
                rev=AUDIO_STACK_ID,
                wavlm=self.model_name,
                wavlm_commit=self.wavlm_commit,
                wavlm_config_fingerprint=self.wavlm_config_fingerprint,
                ecapa=self.ecapa_source,
                ecapa_cache_fingerprint=self.ecapa_cache_fingerprint,
                context_s=float(WAVLM_CONTEXT_S),
                recent_s=float(WAVLM_RECENT_CONTEXT_S),
                ecapa_segment_s=float(ECAPA_SEGMENT_S),
                data_fingerprint=str(MULTITPC_DATA_FINGERPRINT),
                audio_alignment_fingerprint=str(AUDIO_ALIGNMENT_FINGERPRINT),
                alignment_rev=str(ALIGNMENT_CONTRACT_REV),
            )
            self.fp=_support_sha256_json(self.cache_contract,n=24)
            self.cache_dir=AUDIO_FEATURE_CACHE_ROOT/f"general_audio_{self.fp}"
            self.cache_dir.mkdir(parents=True,exist_ok=True)
            self.available=True
        except Exception as exc:
            self.available=False
            self.reason=repr(exc)
            # Release partially loaded models after a failed SpeechBrain load.
            try:
                if self.wavlm is not None:self.wavlm.to("cpu")
            except Exception:pass
            try:
                if self.ecapa is not None and hasattr(self.ecapa,"to"):self.ecapa.to("cpu")
            except Exception:pass
            self.wavlm=None;self.ecapa=None
            release_resources(note="partial causal-audio load released")
            print("Causal WavLM/ECAPA adapter unavailable:",self.reason)

    def _cache_path(self,session,t):
        safe=re.sub(r"[^A-Za-z0-9_.-]+","_",str(session))
        frame=int(round(float(t)*FRAME_HZ))
        return self.cache_dir/f"{safe}_f{frame:08d}.npz"

    @torch.inference_mode()
    def _wavlm_features(self,session,t):
        wave=_read_mixed_audio_history(session,t,WAVLM_CONTEXT_S)
        if len(wave)<320:wave=np.pad(wave,(320-len(wave),0))
        x=torch.as_tensor(wave,dtype=torch.float32,device=self.device).unsqueeze(0)
        with torch.autocast(device_type=self.device.type,dtype=torch.float16,enabled=(self.device.type=="cuda")):
            h=self.wavlm(x).last_hidden_state[0].detach().float().cpu().numpy().astype(np.float32)
        fps=max(1.0,len(h)/max(1e-6,float(WAVLM_CONTEXT_S)))
        n=max(1,int(round(float(WAVLM_RECENT_CONTEXT_S)*fps)))
        recent=h[-n:].mean(axis=0)
        prior=h[-2*n:-n].mean(axis=0) if len(h)>=2*n else h[:-n].mean(axis=0) if len(h)>n else np.zeros(h.shape[1],np.float32)
        whole=h.mean(axis=0)
        d=h.shape[1]
        if self._wavlm_projection is None or self._wavlm_projection.shape!=(d,int(WAVLM_PROJECTION_DIM)):
            rng=np.random.default_rng(stable_seed(f"r11_wavlm_projection::{self.model_name}::{d}"))
            self._wavlm_projection=rng.normal(
                0.0,1.0/np.sqrt(max(1,d)),size=(d,int(WAVLM_PROJECTION_DIM))
            ).astype(np.float32)
        P=self._wavlm_projection
        return np.r_[recent@P,(recent-prior)@P,whole@P].astype(np.float32)

    @torch.inference_mode()
    def _ecapa_embed(self,wave):
        wave=np.asarray(wave,np.float32)
        wave=np.nan_to_num(wave,nan=0.0,posinf=0.0,neginf=0.0)
        if len(wave)<int(ECAPA_MIN_SEGMENT_S*TARGET_SR):return None
        try:
            x=torch.as_tensor(wave,dtype=torch.float32,device=self.device).unsqueeze(0)
            emb=self.ecapa.encode_batch(x).squeeze().detach().float().cpu().numpy()
            return _l2_normalize(emb)
        except Exception as exc:
            raise RuntimeError("ECAPA embedding failed; no silent fallback.") from exc

    def _ecapa_features(self,session,t):
        recent=[];prior=[];energy=[]
        seg=float(ECAPA_SEGMENT_S)
        for p in (1,2,3):
            wr=_raw_audio_interval(str(session),p,float(t)-seg,float(t))
            wp=_raw_audio_interval(str(session),p,float(t)-2*seg,float(t)-seg)
            recent.append(self._ecapa_embed(wr))
            prior.append(self._ecapa_embed(wp))
            er=float(np.sqrt(np.mean(np.asarray(wr,float)**2)+1e-12)) if len(wr) else 0.0
            ep=float(np.sqrt(np.mean(np.asarray(wp,float)**2)+1e-12)) if len(wp) else 0.0
            energy.extend([np.log1p(er),np.log1p(ep),np.log1p(er)-np.log1p(ep)])
        feat=[];names=[]
        for p in range(3):
            c,v=_cosine_with_validity(recent[p],prior[p]);feat += [c,v];names += [f"ecapa_p{p+1}_self_continuity",f"ecapa_p{p+1}_self_valid"]
        for a,b in ((0,1),(0,2),(1,2)):
            c,v=_cosine_with_validity(recent[a],recent[b]);feat += [c,v];names += [f"ecapa_current_cos_p{a+1}p{b+1}",f"ecapa_current_valid_p{a+1}p{b+1}"]
        feat += energy
        names += sum(([f"ecapa_p{p}_recent_logrms",f"ecapa_p{p}_prior_logrms",f"ecapa_p{p}_delta_logrms"] for p in (1,2,3)),[])
        return np.asarray(feat,np.float32),names

    def transform(self,session,t):
        if not self.available:
            raise RuntimeError(f"Causal audio adapter is unavailable: {self.reason}")
        fp=self._cache_path(session,t)
        if fp.exists():
            try:
                with np.load(fp,allow_pickle=False) as d:
                    if str(d["contract_fp"].item())==str(self.fp):
                        return np.asarray(d["feature"],np.float32),[str(x) for x in d["names"].astype(str)]
            except Exception:pass
        wf=self._wavlm_features(session,t)
        ef,en=self._ecapa_features(session,t)
        wn=[f"wavlm_recent_p{j}" for j in range(WAVLM_PROJECTION_DIM)]
        wn += [f"wavlm_delta_p{j}" for j in range(WAVLM_PROJECTION_DIM)]
        wn += [f"wavlm_whole_p{j}" for j in range(WAVLM_PROJECTION_DIM)]
        feat=np.r_[wf,ef].astype(np.float32);names=wn+en
        np.savez_compressed(fp,feature=feat,names=np.asarray(names,dtype="U"),contract_fp=str(self.fp))
        return feat,names

    def identity(self):
        return dict(
            kind="wavlm_ecapa",
            wavlm_model=str(self.model_name),
            wavlm_commit=str(getattr(self,"wavlm_commit","unknown")),
            wavlm_config_fingerprint=str(getattr(self,"wavlm_config_fingerprint","unknown")),
            ecapa_source=str(self.ecapa_source),
            ecapa_cache_fingerprint=str(getattr(self,"ecapa_cache_fingerprint","unknown")),
            feature_contract_fp=str(self.fp),
        )

    def close(self):
        try:
            if self.wavlm is not None:self.wavlm.to("cpu")
        except Exception:pass
        try:
            if self.ecapa is not None and hasattr(self.ecapa,"to"):self.ecapa.to("cpu")
        except Exception:pass
        self.wavlm=None;self.ecapa=None
        release_resources(note="causal audio adapter released")

def build_audio_feature_matrix(rows, adapter, kind):
    feats = []
    names = None
    for i, r in enumerate(rows.itertuples(index=False), start=1):
        f, n = adapter.transform(str(r.session), float(r.time))
        if names is None:
            names = n
        elif names != n:
            raise RuntimeError(f'{kind} audio feature-name drift')
        feats.append(f)
        if i % 100 == 0:
            print(f'{kind} audio features: {i}/{len(rows)}')
    if not feats:
        return (np.zeros((0, 1), np.float32), [])
    return (np.vstack(feats).astype(np.float32), names)

In [12]:
# Partition isolation, cache helpers, and base classifier
_FIT_ALLOWED = contextvars.ContextVar('r12_ablation_fit_allowed', default=None)
SESSION_GROUP = MANIFEST.set_index('session')['group_id'].astype(str).to_dict()
GROUPS = sorted(set(SESSION_GROUP.values()))
if len(GROUPS) != 7:
    raise RuntimeError(f'Expected seven Multi-TPC groups; found {GROUPS}')
TARGET_COLUMNS = ['nearest_event_distance', 'event_time', 'label', 'event_type', 'next_speaker', 'target_offset']

ABLATION_METHODS = (
    METHOD_FULL,
    METHOD_STAGE1_ONLY,
    METHOD_AUDIO_ONLY,
    METHOD_MINUS_GAZE,
    METHOD_MINUS_MOTION,
    METHOD_MINUS_PROSODY,
    METHOD_MINUS_WAVLM,
    METHOD_MINUS_ECAPA,
)

def canonical_json(obj):
    return json.dumps(_support_jsonable_config(obj),indent=2,sort_keys=True,allow_nan=False)

def atomic_bytes(path, raw):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    fd,tmp=tempfile.mkstemp(prefix='.r12_ablation_pending_',dir=path.parent)
    try:
        with os.fdopen(fd,'wb') as f:
            f.write(raw); f.flush(); os.fsync(f.fileno())
        os.replace(tmp,path)
    finally:
        if Path(tmp).exists(): Path(tmp).unlink()

def cache_put(path,obj,contract):
    raw=pickle.dumps(obj,protocol=pickle.HIGHEST_PROTOCOL)
    atomic_bytes(path,raw)
    atomic_bytes(Path(path).with_suffix('.json'),canonical_json({'contract':contract,'sha256':hashlib.sha256(raw).hexdigest()}).encode())

def cache_get(path,contract):
    path=Path(path); meta=path.with_suffix('.json')
    if not path.exists() or not meta.exists(): return None
    record=json.loads(meta.read_text())
    if record['contract'] != _support_jsonable_config(contract): return None
    if _sha256_file(path) != record['sha256']: raise RuntimeError(f'Cache hash mismatch: {path}')
    with path.open('rb') as f: return pickle.load(f)

def sessions_for(groups):
    return sorted(s for s,g in SESSION_GROUP.items() if g in set(map(str,groups)))

def table_for(groups,labels=True):
    q=pd.concat([SESSION_TABLES[s] for s in sessions_for(groups)],ignore_index=True)
    q['group_id']=q['session'].map(SESSION_GROUP)
    return q if labels else q.drop(columns=TARGET_COLUMNS)

@contextmanager
def fitting_partition(groups):
    token=_FIT_ALLOWED.set(frozenset(map(str,groups)))
    try: yield
    finally: _FIT_ALLOWED.reset(token)

def assert_fit_groups(groups):
    allowed=_FIT_ALLOWED.get()
    if allowed is None or not set(map(str,groups)).issubset(allowed):
        raise RuntimeError(f'Fit outside active training partition: {groups}; allowed={allowed}')

def assert_fit_table(table):
    assert_fit_groups({SESSION_GROUP[s] for s in table['session'].astype(str)})

def matrix(table,features):
    if any(c in TARGET_COLUMNS or c in ('session','group_id','time') for c in features):
        raise RuntimeError('Labels or identifiers cannot enter the model matrix.')
    return np.nan_to_num(table[list(features)].to_numpy(np.float32),nan=0.,posinf=0.,neginf=0.)

def row_keys(q):
    return list(zip(q['session'].astype(str),np.rint(q['time'].to_numpy(float)*1e6).astype(np.int64)))

def attach_values(base,source,columns):
    src_keys=row_keys(source); dst_keys=row_keys(base)
    if len(set(src_keys))!=len(src_keys) or len(set(dst_keys))!=len(dst_keys): raise RuntimeError('Duplicate time keys in feature join.')
    lookup={key:i for i,key in enumerate(src_keys)}
    if any(key not in lookup for key in dst_keys): raise RuntimeError('Unmatched feature row.')
    out=base.copy().reset_index(drop=True)
    for col in columns: out[col]=source[col].to_numpy()[[lookup[key] for key in dst_keys]]
    return out

def probabilities(model,X):
    if not len(X): return np.empty(0)
    p=model.predict_proba(X); classes=list(model.classes_)
    return p[:,classes.index(1)] if 1 in classes else np.zeros(len(X))

class ConstantClassifier:
    def __init__(self,value): self.value=int(value); self.classes_=np.array([self.value])
    def predict(self,X): return np.full(len(X),self.value,int)
    def predict_proba(self,X): return np.ones((len(X),1))

def fit_classifier(table, features, target, kind='logistic', param=0.3, tag=''):
    assert_fit_table(table)
    if table.empty: return {'model': ConstantClassifier(0), 'scaler': None, 'features': list(features)}
    y = table[target].to_numpy(int); X = matrix(table, features)
    if len(np.unique(y)) == 1: return {'model': ConstantClassifier(y[0]), 'scaler': None, 'features': list(features)}
    if kind == 'hgb':
        rng = np.random.default_rng(stable_seed(tag))
        if target == 'label':
            pos = np.flatnonzero(y == 1); neg = np.flatnonzero(y == 0)
            keep = np.sort(np.r_[pos, rng.choice(neg, min(len(neg), max(5000, 5*len(pos))), replace=False)])
        else: keep = rng.choice(len(y), min(len(y), 120000), replace=False)
        X = X[keep]; y = y[keep]
        # Disable sklearn's default random-row early-stopping validation split.
        model = HistGradientBoostingClassifier(**param, early_stopping=False, random_state=stable_seed(tag))
        scaler = None
    else:
        scaler = StandardScaler().fit(X); X = scaler.transform(X)
        model = LogisticRegression(C=float(param), max_iter=2000, random_state=stable_seed(tag), class_weight='balanced')
    if kind == 'hgb':
        counts = pd.Series(y).value_counts().to_dict()
        weights = np.array([len(y)/(len(counts)*counts[int(v)]) for v in y])
        model.fit(X, y, sample_weight=weights)
    else: model.fit(X, y)
    return {'model': model, 'scaler': scaler, 'features': list(features)}

def predict_classifier(bundle, table, binary=True):
    if not len(table): return np.empty(0, float if binary else int)
    X = matrix(table, bundle['features'])
    if bundle['scaler'] is not None: X = bundle['scaler'].transform(X)
    return probabilities(bundle['model'], X) if binary else bundle['model'].predict(X)

In [13]:
# Event matching, operating-point calibration, and candidate gating
MATCH_COLUMNS = ['session','pred_time','decision_time','true_time','signed_error_s','abs_error_s','lead_s','true_type','true_next','pred_type','pred_next']
EVENT_COLUMNS = ['session','time','decision_time','score','pred_type','pred_next']

def extract_events(scores, threshold, nms_s, offset_s=0.):
    rows = []
    for session, g in scores.groupby('session', sort=True):
        last = -np.inf
        for r in g.sort_values('time').itertuples(index=False):
            if not np.isfinite(r.score): raise RuntimeError('Nonfinite detection score.')
            if r.score < threshold or r.time-last < nms_s-1e-9: continue
            rows.append({'session':str(session),'time':float(r.time+offset_s),'decision_time':float(r.time),
                         'score':float(r.score),'pred_type':int(getattr(r,'pred_type',-1)),'pred_next':int(getattr(r,'pred_next',-1))})
            last = r.time
    return pd.DataFrame(rows, columns=EVENT_COLUMNS)

def match_events(pred, truth, sessions, tolerance):
    tp = fp = fn = 0; rows = []
    for session in map(str, sessions):
        p = pred[pred.session.astype(str).eq(session)].reset_index(drop=True)
        e = truth[truth.session.astype(str).eq(session)].reset_index(drop=True)
        pairs = []
        if len(p) and len(e):
            cost = np.abs(p.time.to_numpy(float)[:,None] - e.time.to_numpy(float)[None,:])
            rr, cc = linear_sum_assignment(np.where(cost <= tolerance+1e-9, cost, 1e6))
            pairs = [(i,j) for i,j in zip(rr,cc) if cost[i,j] <= tolerance+1e-9]
        tp += len(pairs); fp += len(p)-len(pairs); fn += len(e)-len(pairs)
        for i,j in pairs:
            pr = p.iloc[i]; gt = e.iloc[j]
            rows.append({'session':session,'pred_time':float(pr.time),'decision_time':float(pr.decision_time),
                'true_time':float(gt.time),'signed_error_s':float(pr.time-gt.time),'abs_error_s':float(abs(pr.time-gt.time)),
                'lead_s':float(gt.time-pr.decision_time),'true_type':str(gt.get('event','transfer')),
                'true_next':int(gt.get('next',-1)),'pred_type':int(pr.get('pred_type',-1)),'pred_next':int(pr.get('pred_next',-1))})
    return {'P':tp/max(1,tp+fp),'R':tp/max(1,tp+fn),'F1':2*tp/max(1,2*tp+fp+fn),'TP':int(tp),'FP':int(fp),'FN':int(fn)}, pd.DataFrame(rows,columns=MATCH_COLUMNS)

def select_op(scores, groups):
    assert_fit_groups(groups)
    if not set(scores.session.map(SESSION_GROUP)).issubset(set(groups)): raise RuntimeError('Selection session leakage.')
    finite = scores.score.to_numpy(float)
    thresholds = np.unique(np.r_[0.,0.5,1.000001,np.quantile(finite,SCORE_QUANTILES)]) if len(finite) else np.array([1.000001])
    rows = []
    for threshold, nms, offset in product(thresholds, NMS_GRID_S, OFFSET_GRID_S):
        pred = extract_events(scores, threshold, nms, offset)
        vals = [match_events(pred, EVENTS_ALL, sessions_for([g]), PRIMARY_EVENT_TOLERANCE)[0] for g in groups]
        rows.append({'threshold':float(threshold),'nms_s':float(nms),'offset_s':float(offset),
                     'mean_group_f1':float(np.mean([v['F1'] for v in vals])), 'mean_group_precision':float(np.mean([v['P'] for v in vals]))})
    search = pd.DataFrame(rows).sort_values(['mean_group_f1','mean_group_precision','threshold','nms_s'],ascending=[False,False,False,False],kind='stable')
    return search.iloc[0].to_dict(), search

def binary_metrics_fixed(y, score, threshold):
    """Fixed-threshold binary metrics plus threshold-free AUROC/AUPRC.

    AUROC/AUPRC are intentionally computed from the continuous score, never from
    the thresholded prediction. They are undefined for a single-class target and
    are reported as None in that case.
    """
    y=np.asarray(y,int);score=np.asarray(score,float)
    if len(y)!=len(score): raise RuntimeError('Binary metric length mismatch.')
    if not len(y):
        return {'n':0,'positive_rate':None,'Accuracy':None,'BalancedAccuracy':None,
                'MacroF1':None,'AUROC':None,'AUPRC':None}
    if not np.isfinite(score).all(): raise RuntimeError('Nonfinite binary score encountered.')
    pred=(score>=float(threshold)).astype(np.int8)
    unique=np.unique(y)
    auroc=float(roc_auc_score(y,score)) if len(unique)==2 else None
    auprc=float(average_precision_score(y,score)) if len(unique)==2 else None
    return {'n':int(len(y)),'positive_rate':float(np.mean(y==1)),
            'Accuracy':float(accuracy_score(y,pred)),
            'BalancedAccuracy':float(balanced_accuracy_score(y,pred)),
            'MacroF1':float(f1_score(y,pred,labels=[0,1],average='macro',zero_division=0)),
            'AUROC':auroc,'AUPRC':auprc}

def binary_threshold(y, scores):
    if not len(scores): return 1.000001
    thresholds=np.unique(np.r_[0.,.5,1.000001,np.quantile(scores,np.linspace(.05,.95,37))])
    vals=[(binary_metrics_fixed(y,scores,t)['MacroF1'],float(t)) for t in thresholds]
    return max(vals)[1]

def oracle_points(groups):
    # Diagnostic oracle availability is explicit; it is never used as Task-C input.
    rows=[]
    for s in sessions_for(groups):
        q=SESSION_TABLES[s]; e=EVENTS_ALL[EVENTS_ALL.session.eq(s)]
        negatives=q[q.nearest_event_distance>1.5].time.to_numpy(float)
        rng=np.random.default_rng(stable_seed('oracle::'+s))
        neg=rng.choice(negatives,min(len(negatives),2*len(e)),replace=False)
        for t,y in [(t,1) for t in e.time]+[(t,0) for t in neg]:
            rows.append({'session':s,'time':float(t),'label':y})
    return pd.DataFrame(rows,columns=['session','time','label'])

def sample_past(scores, points, max_age=0.151):
    out=points.copy(); out['score']=0.
    for s,g in points.groupby('session'):
        q=scores[scores.session.eq(s)].sort_values('time')
        if q.empty: continue
        idx=np.searchsorted(q.time.to_numpy(float),g.time.to_numpy(float),side='right')-1
        valid=(idx>=0)&((g.time.to_numpy(float)-q.time.to_numpy(float)[np.maximum(idx,0)])<=max_age)
        out.loc[g.index[valid],'score']=q.score.to_numpy(float)[idx[valid]]
    return out

def candidate_gate(scored):
    # Fixed threshold/refractory rule, identical for all fit and inference partitions.
    p=extract_events(scored[['session','time','score']],DEFAULT_CANDIDATE_THRESHOLD,DEFAULT_CANDIDATE_REFRACTORY_S,0.)
    base=p[['session','decision_time','score']].rename(columns={'decision_time':'time','score':'stage1_score'})
    return attach_values(base,scored,[c for c in scored.columns if c not in ('session','time','score','stage1_score')])

In [14]:
# Feature families and ablation specifications
# Families are inferred from the base feature names.
_audio_preflight=CausalAudioFeatureAdapter()
if not _audio_preflight.available: raise RuntimeError(_audio_preflight.reason)
try:
    AUDIO_IDENTITY=_audio_preflight.identity()
    _,AUDIO_FEATURE_NAMES=_audio_preflight.transform(sessions_for(GROUPS)[0],1.)
finally:
    _audio_preflight.close()

CHEAP_FAMILIES={
    'audio':[n for n in BASE_FEATURE_NAMES if n.startswith('audio_')],
    'gaze':[n for n in BASE_FEATURE_NAMES if n.startswith('gaze_')],
    'prosody':[n for n in BASE_FEATURE_NAMES if n.startswith('prosody_')],
    'motion':[n for n in BASE_FEATURE_NAMES if n.startswith('motion_')],
}
AUDIO_FAMILIES={
    'wavlm':[n for n in AUDIO_FEATURE_NAMES if n.startswith('wavlm_')],
    'ecapa':[n for n in AUDIO_FEATURE_NAMES if n.startswith('ecapa_')],
}
if set(sum(CHEAP_FAMILIES.values(),[])) != set(BASE_FEATURE_NAMES):
    missing=sorted(set(BASE_FEATURE_NAMES)-set(sum(CHEAP_FAMILIES.values(),[])))
    raise RuntimeError(f'Unassigned base features: {missing}')
if set(sum(AUDIO_FAMILIES.values(),[])) != set(AUDIO_FEATURE_NAMES):
    missing=sorted(set(AUDIO_FEATURE_NAMES)-set(sum(AUDIO_FAMILIES.values(),[])))
    raise RuntimeError(f'Unassigned learned-audio features: {missing}')
if any(len(v)==0 for v in CHEAP_FAMILIES.values()) or any(len(v)==0 for v in AUDIO_FAMILIES.values()):
    raise RuntimeError('A required modality family is empty; do not run an invalid ablation.')

# Hyperparameters are selected once on the full configuration and reused by every ablation.
# Each ablation recalibrates only threshold/NMS/offset on inner training folds.
# Audio-only keeps the full Stage-1 candidate pool; its Stage-2 uses only WavLM+ECAPA features.
ABLATION_SPECS={
    METHOD_FULL:dict(
        stage1_remove_cheap=(),stage2_remove_cheap=(),stage2_remove_audio=(),
        stage2_mode='fused',candidate_discovery_contract='full_r12_proposer'),
    METHOD_STAGE1_ONLY:dict(
        stage1_remove_cheap=(),stage2_remove_cheap=(),stage2_remove_audio=('wavlm','ecapa'),
        stage2_mode='none',candidate_discovery_contract='full_r12_proposer_stage1_only'),
    METHOD_AUDIO_ONLY:dict(
        stage1_remove_cheap=(),stage2_remove_cheap=('audio','gaze','prosody','motion'),stage2_remove_audio=(),
        stage2_mode='audio_only',candidate_discovery_contract='full_r12_proposer_shared_candidate_pool'),
    METHOD_MINUS_GAZE:dict(
        stage1_remove_cheap=('gaze',),stage2_remove_cheap=('gaze',),stage2_remove_audio=(),
        stage2_mode='fused',candidate_discovery_contract='ablation_specific_proposer'),
    METHOD_MINUS_MOTION:dict(
        stage1_remove_cheap=('motion',),stage2_remove_cheap=('motion',),stage2_remove_audio=(),
        stage2_mode='fused',candidate_discovery_contract='ablation_specific_proposer'),
    METHOD_MINUS_PROSODY:dict(
        stage1_remove_cheap=('prosody',),stage2_remove_cheap=('prosody',),stage2_remove_audio=(),
        stage2_mode='fused',candidate_discovery_contract='ablation_specific_proposer'),
    METHOD_MINUS_WAVLM:dict(
        stage1_remove_cheap=(),stage2_remove_cheap=(),stage2_remove_audio=('wavlm',),
        stage2_mode='fused',candidate_discovery_contract='full_r12_proposer'),
    METHOD_MINUS_ECAPA:dict(
        stage1_remove_cheap=(),stage2_remove_cheap=(),stage2_remove_audio=('ecapa',),
        stage2_mode='fused',candidate_discovery_contract='full_r12_proposer'),
}
if tuple(ABLATION_SPECS)!=ABLATION_METHODS: raise RuntimeError('Ablation method/spec order drift.')

def stage1_features_for(method):
    spec=ABLATION_SPECS[method]
    removed=set(sum((CHEAP_FAMILIES[k] for k in spec['stage1_remove_cheap']),[]))
    return [n for n in BASE_FEATURE_NAMES if n not in removed]

def stage2_features_for(method):
    spec=ABLATION_SPECS[method]
    if spec['stage2_mode']=='none': return []
    audio_removed=set(sum((AUDIO_FAMILIES[k] for k in spec['stage2_remove_audio']),[]))
    learned=['audio::'+n for n in AUDIO_FEATURE_NAMES if n not in audio_removed]
    if spec['stage2_mode']=='audio_only': return learned
    cheap_removed=set(sum((CHEAP_FAMILIES[k] for k in spec['stage2_remove_cheap']),[]))
    cheap=[n for n in BASE_FEATURE_NAMES if n not in cheap_removed]
    return ['stage1_score']+cheap+learned

# Check the Audio-only and Stage-1-only restrictions.
_EXPECTED_AUDIO_ONLY_STAGE2=['audio::'+n for n in AUDIO_FEATURE_NAMES]
if stage1_features_for(METHOD_AUDIO_ONLY) != stage1_features_for(METHOD_FULL):
    raise RuntimeError('Audio-only must preserve the Full configuration Stage-1 proposer.')
if stage2_features_for(METHOD_AUDIO_ONLY) != _EXPECTED_AUDIO_ONLY_STAGE2:
    raise RuntimeError('Audio-only Stage 2 must contain exactly WavLM+ECAPA learned-audio features.')
if 'stage1_score' in stage2_features_for(METHOD_AUDIO_ONLY) or set(stage2_features_for(METHOD_AUDIO_ONLY)) & set(BASE_FEATURE_NAMES):
    raise RuntimeError('Audio-only Stage 2 leaked stage1_score or cheap causal features.')
if stage2_features_for(METHOD_STAGE1_ONLY):
    raise RuntimeError('Cheap-only must disable Stage 2 completely.')

FEATURE_CONTRACT_ROWS=[]
for method in ABLATION_METHODS:
    spec=ABLATION_SPECS[method]; s1=stage1_features_for(method); s2=stage2_features_for(method)
    FEATURE_CONTRACT_ROWS.append({
        'method':method,'n_stage1_features':len(s1),'n_stage2_features':len(s2),
        'stage1_sha256':_support_sha256_json(s1,32),'stage2_sha256':_support_sha256_json(s2,32),
        'stage1_remove_cheap':'|'.join(spec['stage1_remove_cheap']),
        'stage2_remove_cheap':'|'.join(spec['stage2_remove_cheap']),
        'stage2_remove_audio':'|'.join(spec['stage2_remove_audio']),
        'stage2_mode':spec['stage2_mode'],
        'candidate_discovery_contract':spec['candidate_discovery_contract'],
        'stage2_includes_stage1_score':bool('stage1_score' in s2),
    })
ABLATION_FEATURE_CONTRACT=pd.DataFrame(FEATURE_CONTRACT_ROWS)
display(ABLATION_FEATURE_CONTRACT)
print('Cheap feature family sizes:',{k:len(v) for k,v in CHEAP_FAMILIES.items()})
print('Learned audio family sizes:',{k:len(v) for k,v in AUDIO_FAMILIES.items()})

<PYTHON_ENV>/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[resource cleanup] before R11 WavLM-large/ECAPA ablation: free CUDA=14.54 GiB


Loading weights: 100%|██████████| 488/488 [00:00<00:00, 80302.10it/s]
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


[resource cleanup] R11 general audio adapter released: free CUDA=14.44 GiB


,method,n_stage1_features,n_stage2_features,stage1_sha256,stage2_sha256,stage1_remove_cheap,stage2_remove_cheap,stage2_remove_audio,stage2_mode,candidate_discovery_contract,stage2_includes_stage1_score
0,Full configuration,112,278,933754fe33b001e6e81d6dec14332a65,1f3f6077769ed13c8b530b43272d087b,,,,fused,full_r12_proposer,True
1,Stage-1 only,112,0,933754fe33b001e6e81d6dec14332a65,4f53cda18c2baa0c0354bb5f9a3ecbe5,,,wavlm|ecapa,none,full_r12_proposer_stage1_only,False
2,Audio only,112,165,933754fe33b001e6e81d6dec14332a65,0720dabee7d8fc1bd5a3852a6f8d6600,,audio|gaze|prosody|motion,,audio_only,full_r12_proposer_shared_candidate_pool,False
3,Minus gaze,109,275,83a8ec08a9eecf6d286df757506cfc08,738180f21bbc158acec275a7cd001036,gaze,gaze,,fused,ablation_specific_proposer,True
4,Minus motion,100,266,d94d39670cef1d9bec2b61e48924e60b,2085255471d95685e62f7c8d7e6a06f3,motion,motion,,fused,ablation_specific_proposer,True
5,Minus prosody,105,271,ceb818643b3fe1653b80d5c4baf19536,c3dcdaed48a71cf9df253bac51ac2b3b,prosody,prosody,,fused,ablation_specific_proposer,True
6,Minus WavLM,112,134,933754fe33b001e6e81d6dec14332a65,811b410c41f014f44f624d700199a888,,,wavlm,fused,full_r12_proposer,True
7,Minus ECAPA,112,257,933754fe33b001e6e81d6dec14332a65,34f0c6164937e253bf09a2781d861aab,,,ecapa,fused,full_r12_proposer,True


Cheap feature family sizes: {'audio': 90, 'gaze': 3, 'prosody': 7, 'motion': 12}
Learned audio family sizes: {'wavlm': 144, 'ecapa': 21}


In [15]:
# Fixed-configuration fitting and scoring
def get_stage1(groups,config_index,features):
    assert_fit_groups(groups)
    contract={'kind':'ablation_stage1','groups':sorted(groups),'config':int(config_index),'features':list(features),'run':RUN_FINGERPRINT}
    path=MODEL_CACHE/('stage1_'+_support_sha256_json(contract)+'.pkl')
    bundle=cache_get(path,contract)
    if bundle is None:
        q=table_for(groups)
        bundle=fit_classifier(q,features,'label','hgb',STAGE1_MODEL_CONFIGS[config_index],str(contract))
        cache_put(path,bundle,contract)
    return bundle

def audio_features(rows,adapter):
    q=rows[['session','time']].copy().reset_index(drop=True)
    names=AUDIO_FEATURE_NAMES
    if len(q):
        values,actual=build_audio_feature_matrix(q,adapter,'cached causal acoustic features')
        if actual!=names: raise RuntimeError('Audio feature contract drift.')
    else: values=np.empty((0,len(names)),np.float32)
    out=rows.copy().reset_index(drop=True)
    return pd.concat([out,pd.DataFrame(values,columns=['audio::'+n for n in names])],axis=1)

def fit_ablation_partition(groups,config_index,stage2_C,method,adapter):
    assert_fit_groups(groups)
    s1_features=stage1_features_for(method); s2_features=stage2_features_for(method)
    contract={'kind':'ablation_pipeline','groups':sorted(groups),'config':int(config_index),'C':float(stage2_C),
              'method':method,'s1':s1_features,'s2':s2_features,'run':RUN_FINGERPRINT}
    path=MODEL_CACHE/('pipeline_'+_support_sha256_json(contract)+'.pkl')
    saved=cache_get(path,contract)
    if saved is not None: return saved
    final_stage1=get_stage1(groups,config_index,s1_features)
    if ABLATION_SPECS[method]['stage2_mode']=='none':
        saved={'groups':sorted(groups),'method':method,'stage1':final_stage1,'stage2':None,'heads':{},'stage1_features':s1_features,'stage2_features':[]}
        cache_put(path,saved,contract); return saved

    parts=[]
    # Candidate training is cross-fitted inside the active fit partition. No held/evaluation group can enter Stage 2.
    for held in sorted(groups):
        fit_groups=[g for g in groups if g!=held]
        stage1=get_stage1(fit_groups,config_index,s1_features)
        q=table_for([held]); q['score']=predict_classifier(stage1,q)
        parts.append(candidate_gate(q))
    train=audio_features(pd.concat(parts,ignore_index=True),adapter)
    if train.empty: raise RuntimeError(f'No training candidates for {method}.')
    stage2=fit_classifier(train,s2_features,'label',param=stage2_C,tag=f'{contract}::stage2')
    pos=train[train.label.eq(1)]
    heads={name:fit_classifier(pos,s2_features,name,param=.3,tag=f'{contract}::{name}') for name in ('event_type','next_speaker')}
    saved={'groups':sorted(groups),'method':method,'stage1':final_stage1,'stage2':stage2,'heads':heads,
           'stage1_features':s1_features,'stage2_features':s2_features}
    cache_put(path,saved,contract)
    return saved

def score_ablation_partition(bundle,groups,adapter):
    if set(map(str,groups)) & set(map(str,bundle['groups'])): raise RuntimeError('Evaluation group occurs in fitted pipeline.')
    dense=table_for(groups,labels=False); dense['score']=predict_classifier(bundle['stage1'],dense)
    if bundle['stage2'] is None:
        scores=dense[['session','time','score']].copy(); scores['pred_type']=-1; scores['pred_next']=-1
        return {'dense':dense[['session','time','score']].copy(),'scores':scores,'candidate_pool':candidate_points(dense)}
    candidates=audio_features(candidate_gate(dense),adapter)
    q=candidates[['session','time']].copy(); q['score']=predict_classifier(bundle['stage2'],candidates)
    q['pred_type']=predict_classifier(bundle['heads']['event_type'],candidates,binary=False)
    q['pred_next']=predict_classifier(bundle['heads']['next_speaker'],candidates,binary=False)
    return {'dense':dense[['session','time','score']].copy(),'scores':q,'candidate_pool':candidate_points(dense)}

# Recovers full-model hyperparameters when the seven-group run is unavailable.
def fit_full_search_partition(groups,config_index,adapter):
    assert_fit_groups(groups)
    s1=stage1_features_for(METHOD_FULL)
    contract={'kind':'full_search','groups':sorted(groups),'config':int(config_index),'run':RUN_FINGERPRINT}
    path=MODEL_CACHE/('full_search_'+_support_sha256_json(contract)+'.pkl')
    saved=cache_get(path,contract)
    if saved is not None: return saved
    parts=[]
    for held in sorted(groups):
        fit_groups=[g for g in groups if g!=held]
        stage1=get_stage1(fit_groups,config_index,s1)
        q=table_for([held]); q['score']=predict_classifier(stage1,q); parts.append(candidate_gate(q))
    train=audio_features(pd.concat(parts,ignore_index=True),adapter)
    s2=stage2_features_for(METHOD_FULL)
    models={float(C):fit_classifier(train,s2,'label',param=C,tag=f'{contract}::{C}') for C in STAGE2_REGULARIZATION_GRID}
    saved={'groups':sorted(groups),'stage1':get_stage1(groups,config_index,s1),'models':models,'stage2_features':s2}
    cache_put(path,saved,contract); return saved

def score_full_search_partition(bundle,groups,adapter):
    if set(map(str,groups)) & set(map(str,bundle['groups'])): raise RuntimeError('Evaluation group occurs in fitted full-search pipeline.')
    dense=table_for(groups,labels=False); dense['score']=predict_classifier(bundle['stage1'],dense)
    candidates=audio_features(candidate_gate(dense),adapter)
    out={'cheap':dense[['session','time','score']].copy()}
    for C,model in bundle['models'].items():
        q=candidates[['session','time']].copy(); q['score']=predict_classifier(model,candidates); out[float(C)]=q
    return out

In [16]:
# Run contract, optional seven-group selection reuse, and immutable commits
BASE_REFERENCE_NOTEBOOK = {
    'public_name': 'multitpc_seven_group_evaluation.ipynb',
    'original_source_name': 'MultiTPC_R12_GROUP_NESTED_DEVELOPMENT_AND_VOX_TRANSFER_v3_FULLY_AUDITED.ipynb',
    'original_sha256': 'a3af62e735e3d8066d67a02b12897c6edceff2b59a4e53338dcf279db827ecd8',
    'legacy_full_method_key': 'R12_audio_fused',
}
LEGACY_FULL_METHOD_KEY = BASE_REFERENCE_NOTEBOOK['legacy_full_method_key']

PACKAGE_VERSIONS={p:importlib.metadata.version(p) for p in ('numpy','pandas','scipy','scikit-learn','torch','transformers','speechbrain')}
RUN_CONTRACT={
    'protocol':ABLATION_PROTOCOL,'reporting_revision':ABLATION_REPORTING_REV,'implementation_revision':ABLATION_IMPLEMENTATION_REV,
    'base_r12_protocol':BASE_PROTOCOL_ID,'base_implementation_sha256':IMPLEMENTATION_SHA256,
    'base_reference_notebook': BASE_REFERENCE_NOTEBOOK,
    'groups':GROUPS,'data':MULTITPC_DATA_FINGERPRINT,'alignment':AUDIO_ALIGNMENT_FINGERPRINT,'audio':AUDIO_IDENTITY,
    'feature_contract':ABLATION_FEATURE_CONTRACT.to_dict('records'),'packages':PACKAGE_VERSIONS,
    'stage1_configs':STAGE1_MODEL_CONFIGS,'stage2_C':STAGE2_REGULARIZATION_GRID,'event_tolerances':EVENT_TOLERANCES,
    'selection_rule':'Select Stage-1 configuration and Stage-2 regularization using the full configuration only inside each outer fold; reuse them unchanged for every ablation. Recalibrate threshold/NMS/offset on inner folds per ablation.',
    'claim':'Post-hoc fixed-configuration seven-group ablation; explanatory evidence, not untouched confirmatory TEST.'
}
RUN_FINGERPRINT=_support_sha256_json(RUN_CONTRACT,n=32)
RUN_DIR=ABLATION_ROOT/('protocol_'+RUN_FINGERPRINT)
MODEL_ROOT=RUN_DIR/'models'; MODEL_CACHE=MODEL_ROOT/'partition_models'; RESULT_ROOT=RUN_DIR/'results'; FIG_ROOT=RUN_DIR/'figures'; FOLD_ROOT=RUN_DIR/'outer_folds'
for _p in (MODEL_ROOT,MODEL_CACHE,RESULT_ROOT,FIG_ROOT,FOLD_ROOT): _p.mkdir(parents=True,exist_ok=True)
atomic_bytes(RUN_DIR/'protocol.json',canonical_json(RUN_CONTRACT).encode())
print('Run fingerprint:',RUN_FINGERPRINT)

# Reuse the seven-group full-model selection (artifact key `R12_audio_fused.pkl`) when verified;
# otherwise rerun full-only nested selection.
MAIN_FULL_METHOD_NAME = LEGACY_FULL_METHOD_KEY
MAIN_SELECTION_DISCOVERY_LOG=[]

def discover_main_full_selection(outer):
    env = os.environ.get('MULTITPC_SEVEN_GROUP_RUN_DIR', os.environ.get('R12_MAIN_RUN_DIR', '')).strip()
    roots=[Path(env).expanduser()] if env else []
    if BASE_MAIN_ROOT.exists(): roots += sorted(BASE_MAIN_ROOT.glob('protocol_*'))
    # Preserve order while removing duplicate roots.
    unique_roots=[]; seen=set()
    for root in roots:
        key=str(root.resolve()) if root.exists() else str(root)
        if key not in seen: seen.add(key); unique_roots.append(root)
    candidates=[]; rejected=[]
    for root in unique_roots:
        protocol_path=root/'protocol.json'
        try:
            if not protocol_path.is_file():
                rejected.append((str(root),'missing_protocol_json')); continue
            protocol=json.loads(protocol_path.read_text())
            if protocol.get('protocol')!=BASE_PROTOCOL_ID or protocol.get('implementation_sha256')!=IMPLEMENTATION_SHA256:
                rejected.append((str(root),'base_protocol_or_implementation')); continue
            if sorted(map(str,protocol.get('groups',[])))!=sorted(map(str,GROUPS)):
                rejected.append((str(root),'group_contract')); continue
            if protocol.get('data')!=MULTITPC_DATA_FINGERPRINT or protocol.get('alignment')!=AUDIO_ALIGNMENT_FINGERPRINT:
                rejected.append((str(root),'data_or_alignment_contract')); continue
            if protocol.get('audio')!=AUDIO_IDENTITY:
                rejected.append((str(root),'audio_identity_contract')); continue
        except Exception as exc:
            rejected.append((str(root),f'protocol_exception:{type(exc).__name__}')); continue
        p=root/'outer_folds'/str(outer)/(MAIN_FULL_METHOD_NAME+'.pkl')
        m=p.with_suffix('.commit.json')
        if p.is_file() and m.is_file(): candidates.append((root,p,m))
    for root,p,m in reversed(candidates):
        try:
            meta=json.loads(m.read_text())
            if meta.get('method')!=MAIN_FULL_METHOD_NAME or str(meta.get('outer'))!=str(outer):
                rejected.append((str(p),'manifest_identity')); continue
            if meta.get('role')!='posthoc_nested_logo' or meta.get('committed_before_metric_evaluation') is not True:
                rejected.append((str(p),'manifest_role')); continue
            if _sha256_file(p)!=meta.get('sha256'):
                rejected.append((str(p),'sha256')); continue
            with p.open('rb') as f: payload=pickle.load(f)
            train=[g for g in GROUPS if g!=outer]
            if str(payload.get('outer'))!=str(outer) or payload.get('method')!=MAIN_FULL_METHOD_NAME:
                rejected.append((str(p),'payload_identity')); continue
            if sorted(map(str,payload.get('train_groups',[])))!=sorted(map(str,train)):
                rejected.append((str(p),'train_groups')); continue
            sel=payload.get('selection',{}); key=sel.get('key')
            cfg=int(sel['config'])
            if sel.get('method') not in (None,MAIN_FULL_METHOD_NAME):
                rejected.append((str(p),'selection_method')); continue
            if not isinstance(key,(tuple,list)) or len(key)!=2 or str(key[0])!='fused':
                rejected.append((str(p),'selection_key')); continue
            C=float(key[1])
            if cfg not in range(len(STAGE1_MODEL_CONFIGS)) or C not in set(map(float,STAGE2_REGULARIZATION_GRID)):
                rejected.append((str(p),'selection_grid')); continue
            record={'outer':str(outer),'status':'REUSED','source_path':str(p),'config':cfg,'C':C,'rejected_candidates':rejected}
            MAIN_SELECTION_DISCOVERY_LOG.append(record)
            return {'config':cfg,'C':C,'source':'exact_notebook1_commit','source_path':str(p),'source_sha256':_sha256_file(p)}
        except Exception as exc:
            rejected.append((str(p),f'exception:{type(exc).__name__}'))
    MAIN_SELECTION_DISCOVERY_LOG.append({
        'outer':str(outer),'status':'FALLBACK_TO_LOCAL_FULL_SELECTION','searched_roots':[str(r) for r in unique_roots],
        'expected_method':MAIN_FULL_METHOD_NAME,'rejected_candidates':rejected
    })
    return None

def _reset_peak_cuda():
    if torch.cuda.is_available():
        try: torch.cuda.reset_peak_memory_stats()
        except Exception: pass

def _peak_cuda_gb():
    if not torch.cuda.is_available(): return 0.0
    try: return float(torch.cuda.max_memory_allocated())/(1024**3)
    except Exception: return None

def _group_duration_s(group):
    return float(sum(len(SESSION_TABLES[s])*.1 for s in sessions_for([group])))

def commit_outer(outer,method,payload):
    directory=FOLD_ROOT/str(outer); directory.mkdir(parents=True,exist_ok=True)
    path=directory/(method+'.pkl'); manifest=path.with_suffix('.commit.json')
    if path.exists() or manifest.exists(): raise RuntimeError(f'Partial/unverified ablation output exists: {path}. Preserve and investigate; no overwrite allowed.')
    raw=pickle.dumps(payload,protocol=pickle.HIGHEST_PROTOCOL)
    with path.open('xb') as f: f.write(raw); f.flush(); os.fsync(f.fileno())
    record={'run':RUN_FINGERPRINT,'outer':outer,'method':method,'sha256':hashlib.sha256(raw).hexdigest(),
            'prediction_path':str(path),'role':'posthoc_fixed_configuration_ablation','committed_before_metric_evaluation':True}
    with manifest.open('x') as f: f.write(canonical_json(record))
    return payload

def load_outer(outer,method):
    path=FOLD_ROOT/str(outer)/(method+'.pkl'); meta=path.with_suffix('.commit.json')
    if not path.exists() and not meta.exists(): return None
    if not path.is_file() or not meta.is_file(): raise RuntimeError(f'Incomplete outer commit: {path}')
    record=json.loads(meta.read_text())
    if record.get('run')!=RUN_FINGERPRINT or record.get('outer')!=outer or record.get('method')!=method: raise RuntimeError('Outer commit identity mismatch.')
    if _sha256_file(path)!=record['sha256']: raise RuntimeError('Outer prediction bytes changed.')
    with path.open('rb') as f: return pickle.load(f)

Base R12 source parity: {'r12_session_table': True, 'R11GeneralAudioAdapter': True, 'extract_events': True, 'match_events': True, 'select_op': True, 'fit_classifier': True, 'predict_classifier': True, '_raw_audio_interval': True}
Run fingerprint: f0baefbafdc4b25bb3f0a56be3dcb928


In [17]:
# Full-configuration selection and outer-fold ablation execution
def choose_full_hyperparameters(outer,adapter):
    imported=discover_main_full_selection(outer)
    if imported is not None:
        print(f'{outer}: reusing exact Notebook-1 FULL config/C selection:',imported['config'],imported['C'])
        return imported
    print(f'{outer}: no verified Notebook-1 {MAIN_FULL_METHOD_NAME} commit found; reproducing FULL-only nested selection locally.')
    train=[g for g in GROUPS if g!=outer]
    search_rows=[]
    cache={}
    # Select the Stage-1 config on the cheap proposer, then Stage-2 C within it.
    for cfg in range(len(STAGE1_MODEL_CONFIGS)):
        folds=[]
        for held in train:
            fit=[g for g in train if g!=held]
            with fitting_partition(fit): bundle=fit_full_search_partition(fit,cfg,adapter)
            folds.append(score_full_search_partition(bundle,[held],adapter))
        cache[cfg]=folds
        cheap=pd.concat([f['cheap'] for f in folds],ignore_index=True)
        with fitting_partition(train): op,_=select_op(cheap,train)
        search_rows.append({'kind':'cheap','config':cfg,'C':None,'op':op})
    cheap_pick=sorted([r for r in search_rows if r['kind']=='cheap'],key=lambda r:(r['op']['mean_group_f1'],r['op']['mean_group_precision']),reverse=True)[0]
    cfg=int(cheap_pick['config'])
    for C in map(float,STAGE2_REGULARIZATION_GRID):
        fused=pd.concat([f[C] for f in cache[cfg]],ignore_index=True)
        with fitting_partition(train): op,_=select_op(fused,train)
        search_rows.append({'kind':'fused','config':cfg,'C':C,'op':op})
    fused_pick=sorted([r for r in search_rows if r['kind']=='fused'],key=lambda r:(r['op']['mean_group_f1'],r['op']['mean_group_precision']),reverse=True)[0]
    out={'config':cfg,'C':float(fused_pick['C']),'source':'derived_full_only_nested_selection','search':search_rows}
    return out

def candidate_points(dense):
    return candidate_gate(dense)[['session','time']].copy()

def distance_labels(points,tolerance):
    labels=np.zeros(len(points),np.int8)
    for s,g in points.groupby('session'):
        d=_nearest_event_payload(s,g.time.to_numpy(float))['distance']
        labels[g.index]=(d<=tolerance+1e-9)
    return labels

def diagnostic_thresholds(inner_dense,inner_taskb,train_groups):
    assert_fit_groups(train_groups)
    a=sample_past(inner_dense,oracle_points(train_groups)); b=inner_taskb.reset_index(drop=True)
    return {'taskA':binary_threshold(a.label,a.score),
            'taskB':{str(t):binary_threshold(distance_labels(b,t),b.score) for t in EVENT_TOLERANCES}}

def run_ablation_outer(outer,adapter):
    existing={m:load_outer(outer,m) for m in ABLATION_METHODS}
    if all(v is not None for v in existing.values()): return existing
    train=[g for g in GROUPS if g!=outer]
    present=[p for p in existing.values() if p is not None]
    if present:
        cfgs={int(p['selection']['config']) for p in present}; Cs={float(p['selection']['C']) for p in present}
        if len(cfgs)!=1 or len(Cs)!=1:
            raise RuntimeError(f'Existing outer commits disagree on frozen config/C for {outer}: cfg={cfgs}, C={Cs}')
        selection={'config':next(iter(cfgs)),'C':next(iter(Cs)),'source':'resume_from_existing_outer_commit'}
        print(f'{outer}: resume locks config/C to existing immutable commits:',selection['config'],selection['C'])
    else:
        selection=choose_full_hyperparameters(outer,adapter)
    cfg=int(selection['config']); C=float(selection['C'])
    results={}
    for method in ABLATION_METHODS:
        if existing[method] is not None:
            results[method]=existing[method]; continue
        print(f'  {outer} :: {method}')
        _reset_peak_cuda(); t0=time.perf_counter(); inner_scores=[]; inner_dense=[]; inner_taskb=[]
        for held in train:
            fit=[g for g in train if g!=held]
            with fitting_partition(fit): bundle=fit_ablation_partition(fit,cfg,C,method,adapter)
            scored=score_ablation_partition(bundle,[held],adapter)
            inner_scores.append(scored['scores']); inner_dense.append(scored['dense'])
            inner_taskb.append(sample_past(scored['scores'],scored['candidate_pool']))
        oof=pd.concat(inner_scores,ignore_index=True); dense_oof=pd.concat(inner_dense,ignore_index=True); taskb_oof=pd.concat(inner_taskb,ignore_index=True)
        with fitting_partition(train):
            op,_=select_op(oof,train)
            thresholds=diagnostic_thresholds(dense_oof,taskb_oof,train)
        t_refit0=time.perf_counter()
        with fitting_partition(train): final_bundle=fit_ablation_partition(train,cfg,C,method,adapter)
        t_refit1=time.perf_counter(); t_infer0=time.perf_counter()
        scored=score_ablation_partition(final_bundle,[outer],adapter)
        t_infer1=time.perf_counter(); duration=_group_duration_s(outer)
        cost={'device':str(DEVICE),'train_calibration_wall_s':float(t_refit0-t0),'outer_refit_wall_s':float(t_refit1-t_refit0),
              'outer_inference_wall_s':float(t_infer1-t_infer0),'total_outer_wall_s':float(t_infer1-t0),
              'eval_audio_duration_s':duration,'inference_real_time_factor':float((t_infer1-t_infer0)/max(1e-9,duration)),
              'peak_cuda_allocated_gb':_peak_cuda_gb(),'inner_fit_calls':len(train),'outer_refit_calls':1}
        payload={'outer':outer,'method':method,'train_groups':train,
                 'selection':{'config':cfg,'C':C,'source':selection['source'],'op':op},'thresholds':thresholds,
                 'taskA_dense':scored['dense'],'taskB':sample_past(scored['scores'],scored['candidate_pool']),'scores':scored['scores'],
                 'candidate_pool':scored['candidate_pool'],'feature_contract':ABLATION_FEATURE_CONTRACT[ABLATION_FEATURE_CONTRACT.method.eq(method)].iloc[0].to_dict(),
                 'stage1_features':final_bundle['stage1_features'],'stage2_features':final_bundle['stage2_features'],
                 'taskA_definition':'ablation_specific_proposer_at_oracle_points' if method in (METHOD_MINUS_GAZE,METHOD_MINUS_MOTION,METHOD_MINUS_PROSODY) else 'shared_full_proposer_at_oracle_points',
                 'cost':cost,'fit_contract':'All fits and operating-point calibration exclude the outer group; hyperparameters are selected by the full configuration only.'}
        results[method]=commit_outer(outer,method,payload)
    return results

OUTER_PREDICTIONS={}
_adapter=CausalAudioFeatureAdapter()
if not _adapter.available: raise RuntimeError(_adapter.reason)
try:
    for outer in GROUPS:
        print('Ablation outer:',outer)
        for method,p in run_ablation_outer(outer,_adapter).items(): OUTER_PREDICTIONS[(outer,method)]=p
finally:
    _adapter.close()
print(f'Committed {len(GROUPS)*len(ABLATION_METHODS)} outer ablation prediction packages. Evaluation begins only now.')

In [18]:
# Task A/B/C, localisation, subtype, next-speaker, and runtime evaluation
def binary_metrics_fixed(y,score,threshold):
    y=np.asarray(y,int); score=np.asarray(score,float)
    if len(y)!=len(score): raise RuntimeError('Binary metric length mismatch.')
    if not len(y): return {'n':0,'positive_rate':None,'Accuracy':None,'BalancedAccuracy':None,'MacroF1':None,'AUROC':None,'AUPRC':None}
    if not np.isfinite(score).all(): raise RuntimeError('Nonfinite binary score.')
    pred=(score>=float(threshold)).astype(np.int8); unique=np.unique(y)
    return {'n':int(len(y)),'positive_rate':float(np.mean(y==1)),'Accuracy':float(accuracy_score(y,pred)),
            'BalancedAccuracy':float(balanced_accuracy_score(y,pred)),
            'MacroF1':float(f1_score(y,pred,labels=[0,1],average='macro',zero_division=0)),
            'AUROC':float(roc_auc_score(y,score)) if len(unique)==2 else None,
            'AUPRC':float(average_precision_score(y,score)) if len(unique)==2 else None}

taskc=[]; taska=[]; taskb=[]; matches=[]; next_rows=[]; cost_rows=[]
for outer in GROUPS:
    truth=EVENTS_ALL[EVENTS_ALL.session.isin(sessions_for([outer]))]
    duration=sum(len(SESSION_TABLES[s])*.1 for s in sessions_for([outer]))
    for method in ABLATION_METHODS:
        p=load_outer(outer,method)
        if p is None: raise RuntimeError(f'Missing outer ablation package: {outer}/{method}')
        cost_rows.append({'outer_group':outer,'method':method,**p['cost']})
        op=p['selection']['op']; events=extract_events(p['scores'],**{k:op[k] for k in ('threshold','nms_s','offset_s')})
        for tol in EVENT_TOLERANCES:
            metrics,matched=match_events(events,EVENTS_ALL,sessions_for([outer]),tol)
            subtype={}
            for typ,label in (('turn_taking','NTT'),('interruption','ITT')):
                denominator=int(truth.event.eq(typ).sum()); hits=int(matched.true_type.eq(typ).sum())
                subtype[label+'_recall']=hits/denominator if denominator else None
            if len(matched):
                signed=matched.signed_error_s.to_numpy(float)
                loc_mae=float(matched.abs_error_s.mean()); loc_median=float(matched.abs_error_s.median())
                early_count=int((signed<0).sum()); late_count=int((signed>0).sum()); exact_count=int(np.isclose(signed,0.,atol=1e-9).sum())
                early_fraction=float((signed<0).mean()); late_fraction=float((signed>0).mean())
            else:
                loc_mae=loc_median=early_fraction=late_fraction=None; early_count=late_count=exact_count=0
            metrics.update({'predictions_per_min':len(events)/max(1e-9,duration/60.),
                            'localization_mae_s':loc_mae,'localization_median_abs_error_s':loc_median,
                            'early_count':early_count,'late_count':late_count,'exact_count':exact_count,
                            'early_fraction':early_fraction,'late_fraction':late_fraction,**subtype})
            taskc.append({'outer_group':outer,'method':method,'tolerance':tol,**metrics})
            matched['method']=method; matched['outer_group']=outer; matched['tolerance']=tol; matches.append(matched)
            if tol==PRIMARY_EVENT_TOLERANCE:
                eligible=matched[matched.true_next.between(1,3)]; supported=eligible[eligible.pred_next.between(1,3)]
                next_rows.append({'outer_group':outer,'method':method,'n_matched':len(eligible),'n_supported':len(supported),
                                  'accuracy':float(accuracy_score(supported.true_next,supported.pred_next)) if len(supported) else None,
                                  'MacroF1':float(f1_score(supported.true_next,supported.pred_next,labels=[1,2,3],average='macro',zero_division=0)) if len(supported) else None,
                                  'coverage':len(supported)/len(eligible) if len(eligible) else None})
        a=sample_past(p['taskA_dense'],oracle_points([outer]))
        taska.append({'outer_group':outer,'method':method,'definition':p['taskA_definition'],**binary_metrics_fixed(a.label,a.score,p['thresholds']['taskA'])})
        b=p['taskB'].reset_index(drop=True)
        for tol in EVENT_TOLERANCES:
            taskb.append({'outer_group':outer,'method':method,'tolerance':tol,**binary_metrics_fixed(distance_labels(b,tol),b.score,p['thresholds']['taskB'][str(tol)])})

ABL_TASKC=pd.DataFrame(taskc); ABL_TASKA=pd.DataFrame(taska); ABL_TASKB=pd.DataFrame(taskb)
ABL_MATCHES=pd.concat(matches,ignore_index=True) if matches else pd.DataFrame(columns=MATCH_COLUMNS+['method','outer_group','tolerance'])
ABL_NEXT=pd.DataFrame(next_rows); ABL_COST=pd.DataFrame(cost_rows)
for name,table in {'taskC_folds':ABL_TASKC,'taskA_folds':ABL_TASKA,'taskB_folds':ABL_TASKB,'next_speaker_folds':ABL_NEXT,'cost_folds':ABL_COST,'matched_events':ABL_MATCHES}.items():
    table.to_csv(RESULT_ROOT/(name+'.csv'),index=False)
print('Outer-fold metric rows:',len(ABL_TASKC))

Outer-fold metric rows: 168


In [19]:
# Group summaries, paired deltas, bootstrap intervals, and corrected tests
def group_mean_sd_ci(values,tag):
    x=np.asarray(values,float); x=x[np.isfinite(x)]
    if not len(x): return {'mean':None,'sd':None,'ci_low':None,'ci_high':None,'n_groups':0}
    rng=np.random.default_rng(stable_seed(tag)); idx=rng.integers(0,len(x),size=(N_BOOTSTRAP,len(x))); means=x[idx].mean(1)
    return {'mean':float(x.mean()),'sd':float(x.std(ddof=1)) if len(x)>1 else 0.,
            'ci_low':float(np.quantile(means,.025)),'ci_high':float(np.quantile(means,.975)),'n_groups':int(len(x))}

def summarize_taskc():
    rows=[]
    for (method,tol),q in ABL_TASKC.groupby(['method','tolerance']):
        row={'method':method,'tolerance':float(tol),'n_groups':len(q)}
        for metric in ('P','R','F1','predictions_per_min','localization_mae_s','localization_median_abs_error_s','NTT_recall','ITT_recall'):
            s=group_mean_sd_ci(q[metric],f'C:{method}:{tol}:{metric}')
            row.update({metric+'_mean':s['mean'],metric+'_sd':s['sd'],metric+'_ci_low':s['ci_low'],metric+'_ci_high':s['ci_high']})
        tp,fp,fn=q[['TP','FP','FN']].sum(); row.update({'TP_pooled':int(tp),'FP_pooled':int(fp),'FN_pooled':int(fn),
            'P_pooled':float(tp/max(1,tp+fp)),'R_pooled':float(tp/max(1,tp+fn)),'F1_pooled':float(2*tp/max(1,2*tp+fp+fn))})
        rows.append(row)
    return pd.DataFrame(rows)

def summarize_binary(table,keys,prefix):
    rows=[]
    for group,q in table.groupby(keys,dropna=False):
        if not isinstance(group,tuple): group=(group,)
        row=dict(zip(keys,group)); row['n_groups']=len(q)
        for metric in ('MacroF1','AUROC','AUPRC','Accuracy','BalancedAccuracy'):
            s=group_mean_sd_ci(q[metric],f'{prefix}:{group}:{metric}')
            row.update({metric+'_mean':s['mean'],metric+'_sd':s['sd'],metric+'_ci_low':s['ci_low'],metric+'_ci_high':s['ci_high']})
        rows.append(row)
    return pd.DataFrame(rows)

ABL_TASKC_SUMMARY=summarize_taskc()
ABL_TASKA_SUMMARY=summarize_binary(ABL_TASKA,['method'],'A')
ABL_TASKB_SUMMARY=summarize_binary(ABL_TASKB,['method','tolerance'],'B')

paired=[]
for method in ABLATION_METHODS:
    if method==METHOD_FULL: continue
    a=ABL_TASKA[ABL_TASKA.method.eq(METHOD_FULL)][['outer_group','MacroF1']].rename(columns={'MacroF1':'full'})
    b=ABL_TASKA[ABL_TASKA.method.eq(method)][['outer_group','MacroF1']].rename(columns={'MacroF1':'ablated'})
    for r in a.merge(b,on='outer_group',validate='one_to_one').itertuples(index=False): paired.append({'outer_group':r.outer_group,'method':method,'metric':'TaskA_MacroF1','tolerance':np.nan,'full':r.full,'ablated':r.ablated,'delta_full_minus_ablation':r.full-r.ablated})
    for tol in EVENT_TOLERANCES:
        a=ABL_TASKB[(ABL_TASKB.method.eq(METHOD_FULL))&np.isclose(ABL_TASKB.tolerance,tol)][['outer_group','MacroF1']].rename(columns={'MacroF1':'full'})
        b=ABL_TASKB[(ABL_TASKB.method.eq(method))&np.isclose(ABL_TASKB.tolerance,tol)][['outer_group','MacroF1']].rename(columns={'MacroF1':'ablated'})
        for r in a.merge(b,on='outer_group',validate='one_to_one').itertuples(index=False): paired.append({'outer_group':r.outer_group,'method':method,'metric':'TaskB_MacroF1','tolerance':tol,'full':r.full,'ablated':r.ablated,'delta_full_minus_ablation':r.full-r.ablated})
        a=ABL_TASKC[(ABL_TASKC.method.eq(METHOD_FULL))&np.isclose(ABL_TASKC.tolerance,tol)][['outer_group','F1']].rename(columns={'F1':'full'})
        b=ABL_TASKC[(ABL_TASKC.method.eq(method))&np.isclose(ABL_TASKC.tolerance,tol)][['outer_group','F1']].rename(columns={'F1':'ablated'})
        for r in a.merge(b,on='outer_group',validate='one_to_one').itertuples(index=False): paired.append({'outer_group':r.outer_group,'method':method,'metric':'TaskC_F1','tolerance':tol,'full':r.full,'ablated':r.ablated,'delta_full_minus_ablation':r.full-r.ablated})
ABL_PAIRED=pd.DataFrame(paired)

def wilcoxon_safe(x):
    d=np.asarray(x,float); d=d[np.isfinite(d)]
    if not len(d): return None,None
    if np.all(np.isclose(d,0.,atol=1e-12)): return 0.,1.
    try:
        r=wilcoxon(d,zero_method='wilcox',alternative='two-sided',method='auto'); return float(r.statistic),float(r.pvalue)
    except ValueError: return None,None

def holm_adjust(pvals):
    p=np.asarray(pvals,float); out=np.full(len(p),np.nan); finite=np.flatnonzero(np.isfinite(p))
    if not len(finite): return out
    order=finite[np.argsort(p[finite])]; m=len(order); running=0.
    for rank,idx in enumerate(order):
        val=min(1.,(m-rank)*p[idx]); running=max(running,val); out[idx]=running
    return out

tests=[]
for keys,q in ABL_PAIRED.groupby(['method','metric','tolerance'],dropna=False):
    method,metric,tol=keys; d=q.delta_full_minus_ablation.to_numpy(float); stat,p=wilcoxon_safe(d); s=group_mean_sd_ci(d,f'paired:{method}:{metric}:{tol}')
    tests.append({'method':method,'metric':metric,'tolerance':None if pd.isna(tol) else float(tol),'n_groups':len(d),
                  'mean_delta':s['mean'],'sd_delta':s['sd'],'median_delta':float(np.median(d)),'delta_ci_low':s['ci_low'],'delta_ci_high':s['ci_high'],
                  'wilcoxon_statistic':stat,'p_raw':p})
ABL_TESTS=pd.DataFrame(tests); ABL_TESTS['p_holm']=holm_adjust(ABL_TESTS.p_raw.to_numpy(float))

# Positive delta: Full outperforms the ablated condition.
paper=[]
for method in ABLATION_METHODS:
    row={'Method':method}
    qa=ABL_TASKA_SUMMARY[ABL_TASKA_SUMMARY.method.eq(method)].iloc[0]
    row['Task-A Macro-F1']=f"{qa.MacroF1_mean:.3f} ± {qa.MacroF1_sd:.3f}"
    qb=ABL_TASKB_SUMMARY[(ABL_TASKB_SUMMARY.method.eq(method))&np.isclose(ABL_TASKB_SUMMARY.tolerance,PRIMARY_EVENT_TOLERANCE)].iloc[0]
    row['Task-B Macro-F1']=f"{qb.MacroF1_mean:.3f} ± {qb.MacroF1_sd:.3f}"
    for tol in EVENT_TOLERANCES:
        qc=ABL_TASKC_SUMMARY[(ABL_TASKC_SUMMARY.method.eq(method))&np.isclose(ABL_TASKC_SUMMARY.tolerance,tol)].iloc[0]
        row[f'Task-C F1 ±{tol:.1f}']=f"{qc.F1_mean:.3f} ± {qc.F1_sd:.3f}"
    if method==METHOD_FULL: row['Δ Full−Ablated @±1.0']='—'
    else:
        q=ABL_TESTS[(ABL_TESTS.method.eq(method))&(ABL_TESTS.metric.eq('TaskC_F1'))&np.isclose(ABL_TESTS.tolerance.astype(float),1.0)].iloc[0]
        row['Δ Full−Ablated @±1.0']=f"{q.mean_delta:+.3f}"
    paper.append(row)
ABLATION_ARTICLE_TABLE=pd.DataFrame(paper)

for name,table in {'taskC_summary':ABL_TASKC_SUMMARY,'taskA_summary':ABL_TASKA_SUMMARY,'taskB_summary':ABL_TASKB_SUMMARY,
                   'paired_differences':ABL_PAIRED,'paired_tests':ABL_TESTS,'article_ablation_table':ABLATION_ARTICLE_TABLE,
                   'feature_contract':ABLATION_FEATURE_CONTRACT}.items(): table.to_csv(RESULT_ROOT/(name+'.csv'),index=False)
display(ABLATION_ARTICLE_TABLE)
display(ABL_TESTS.sort_values(['metric','tolerance','method']))

,Method,Task-A Macro-F1,Task-B Macro-F1,Task-C F1 ±0.5,Task-C F1 ±0.7,Task-C F1 ±1.0,Δ Full−Ablated @±1.0
0,Full configuration,0.580 ± 0.039,0.604 ± 0.024,0.342 ± 0.037,0.442 ± 0.043,0.510 ± 0.048,—
1,Stage-1 only,0.580 ± 0.039,0.572 ± 0.033,0.316 ± 0.043,0.409 ± 0.039,0.470 ± 0.048,+0.040
2,Audio only,0.580 ± 0.039,0.605 ± 0.019,0.328 ± 0.042,0.429 ± 0.043,0.498 ± 0.043,+0.012
3,Minus gaze,0.578 ± 0.047,0.607 ± 0.020,0.342 ± 0.044,0.441 ± 0.049,0.514 ± 0.050,-0.004
4,Minus motion,0.564 ± 0.066,0.604 ± 0.018,0.339 ± 0.040,0.430 ± 0.044,0.496 ± 0.054,+0.014
5,Minus prosody,0.565 ± 0.044,0.610 ± 0.017,0.346 ± 0.043,0.439 ± 0.046,0.509 ± 0.046,+0.001
6,Minus WavLM,0.580 ± 0.039,0.598 ± 0.028,0.334 ± 0.037,0.440 ± 0.046,0.514 ± 0.052,-0.004
7,Minus ECAPA,0.580 ± 0.039,0.584 ± 0.022,0.337 ± 0.037,0.430 ± 0.046,0.481 ± 0.052,+0.029


,method,metric,tolerance,n_groups,mean_delta,sd_delta,median_delta,delta_ci_low,delta_ci_high,wilcoxon_statistic,p_raw,p_holm
0,Audio only,TaskA_MacroF1,NaN,7,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.000000,1.000000
7,Stage-1 only,TaskA_MacroF1,NaN,7,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.000000,1.000000
14,Minus ECAPA,TaskA_MacroF1,NaN,7,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.000000,1.000000
21,Minus gaze,TaskA_MacroF1,NaN,7,0.002637,0.008882,-0.000460,-0.001812,0.009554,14.0,1.000000,1.000000
28,Minus motion,TaskA_MacroF1,NaN,7,0.016194,0.032792,0.001315,-0.002456,0.041275,9.0,0.468750,1.000000
35,Minus prosody,TaskA_MacroF1,NaN,7,0.015252,0.030896,0.007206,-0.002614,0.038402,7.0,0.296875,1.000000
42,Minus WavLM,TaskA_MacroF1,NaN,7,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.000000,1.000000
1,Audio only,TaskB_MacroF1,0.5,7,-0.001731,0.011822,0.002012,-0.010527,0.005091,12.0,0.812500,1.000000
8,Stage-1 only,TaskB_MacroF1,0.5,7,0.025331,0.032929,0.019614,0.003610,0.048658,3.0,0.078125,1.000000
15,Minus ECAPA,TaskB_MacroF1,0.5,7,0.012736,0.015987,0.015210,0.000662,0.022165,5.0,0.156250,1.000000


In [20]:
# Paper figures from all seven outer groups
sns.set_theme(style='whitegrid',context='paper')
FIGURE_FILES=[]
LABELS={
    METHOD_FULL:'Full configuration',METHOD_STAGE1_ONLY:'Cheap only',METHOD_AUDIO_ONLY:'Audio only',METHOD_MINUS_GAZE:'− Gaze',
    METHOD_MINUS_MOTION:'− Motion',METHOD_MINUS_PROSODY:'− Prosody',METHOD_MINUS_WAVLM:'− WavLM',METHOD_MINUS_ECAPA:'− ECAPA'
}
order=list(ABLATION_METHODS)

def savefig(name):
    for ext in ('png','pdf'):
        p=FIG_ROOT/f'{name}.{ext}'; plt.savefig(p,bbox_inches='tight',dpi=300); FIGURE_FILES.append(str(p))
    plt.close()

q=ABL_TASKC[np.isclose(ABL_TASKC.tolerance,PRIMARY_EVENT_TOLERANCE)].copy(); q['label']=q.method.map(LABELS)
plt.figure(figsize=(11,5)); sns.boxplot(data=q,x='label',y='F1',order=[LABELS[m] for m in order],showfliers=False)
sns.stripplot(data=q,x='label',y='F1',order=[LABELS[m] for m in order],jitter=.12,size=5)
plt.xlabel('Ablation condition'); plt.ylabel('Task-C F1 (±1.0 s)'); plt.xticks(rotation=25,ha='right'); plt.title('Multi-TPC fixed-configuration ablations across seven outer groups'); savefig('ablation_taskC_f1_boxpoints')

q=ABL_PAIRED[(ABL_PAIRED.metric.eq('TaskC_F1'))&np.isclose(ABL_PAIRED.tolerance.astype(float),1.0)].copy(); q['label']=q.method.map(LABELS)
plt.figure(figsize=(10,5)); sns.boxplot(data=q,x='label',y='delta_full_minus_ablation',order=[LABELS[m] for m in order if m!=METHOD_FULL],showfliers=False)
sns.stripplot(data=q,x='label',y='delta_full_minus_ablation',order=[LABELS[m] for m in order if m!=METHOD_FULL],jitter=.12,size=5)
plt.axhline(0,linewidth=1); plt.xlabel('Removed/restricted component'); plt.ylabel('F1(Full) − F1(Ablated)'); plt.xticks(rotation=25,ha='right'); plt.title('Paired outer-group contribution at ±1.0 s'); savefig('ablation_taskC_paired_deltas')

q=ABL_TASKA.copy(); q['label']=q.method.map(LABELS)
plt.figure(figsize=(11,5)); sns.boxplot(data=q,x='label',y='MacroF1',order=[LABELS[m] for m in order],showfliers=False)
sns.stripplot(data=q,x='label',y='MacroF1',order=[LABELS[m] for m in order],jitter=.12,size=5)
plt.xlabel('Ablation condition'); plt.ylabel('Task-A Macro-F1'); plt.xticks(rotation=25,ha='right'); plt.title('Task-A proposer diagnostic'); savefig('ablation_taskA_macroF1_boxpoints')

q=ABL_TASKB[np.isclose(ABL_TASKB.tolerance,PRIMARY_EVENT_TOLERANCE)].copy(); q['label']=q.method.map(LABELS)
plt.figure(figsize=(11,5)); sns.boxplot(data=q,x='label',y='MacroF1',order=[LABELS[m] for m in order],showfliers=False)
sns.stripplot(data=q,x='label',y='MacroF1',order=[LABELS[m] for m in order],jitter=.12,size=5)
plt.xlabel('Ablation condition'); plt.ylabel('Task-B Macro-F1 (±1.0 s label)'); plt.xticks(rotation=25,ha='right'); plt.title('Candidate-level SHIFT/HOLD diagnostic'); savefig('ablation_taskB_macroF1_boxpoints')
print('Figures:',len(FIGURE_FILES),FIG_ROOT)

  artists = ax.bxp(**boxplot_kws)
  artists = ax.bxp(**boxplot_kws)
  artists = ax.bxp(**boxplot_kws)
  artists = ax.bxp(**boxplot_kws)


Figures: 8 <MULTITPC_ROOT>/runs/MultiTPC_R12_FIXED_CONFIGURATION_ABLATIONS_7FOLD/protocol_f0baefbafdc4b25bb3f0a56be3dcb928/figures


In [21]:
# Final checks and result package
audit=[]
def check(name,condition,detail=''):
    audit.append({'check':name,'status':'PASS' if bool(condition) else 'FAIL','detail':str(detail)})

check('seven_outer_groups',len(GROUPS)==7,GROUPS)
check('eight_ablation_conditions',len(ABLATION_METHODS)==8,ABLATION_METHODS)
check('feature_family_partition_complete',
      set(sum(CHEAP_FAMILIES.values(),[]))==set(BASE_FEATURE_NAMES)
      and sum(map(len,CHEAP_FAMILIES.values()))==len(BASE_FEATURE_NAMES)
      and set(sum(AUDIO_FAMILIES.values(),[]))==set(AUDIO_FEATURE_NAMES)
      and sum(map(len,AUDIO_FAMILIES.values()))==len(AUDIO_FEATURE_NAMES))
check('prosody_is_distinct_nonempty_family',len(CHEAP_FAMILIES['prosody'])>0,len(CHEAP_FAMILIES['prosody']))
check('base_reference_notebook_sha_recorded', len(BASE_REFERENCE_NOTEBOOK.get('original_sha256','')) == 64, BASE_REFERENCE_NOTEBOOK)
check('main_full_method_key_verified', MAIN_FULL_METHOD_NAME == LEGACY_FULL_METHOD_KEY, MAIN_FULL_METHOD_NAME)

# Check the Audio-only and Stage-1-only restrictions before reading committed packages.
_expected_audio_only_stage2=['audio::'+n for n in AUDIO_FEATURE_NAMES]
check('audio_only_preserves_full_stage1',
      stage1_features_for(METHOD_AUDIO_ONLY)==stage1_features_for(METHOD_FULL))
check('audio_only_stage2_is_exact_learned_audio_only',
      stage2_features_for(METHOD_AUDIO_ONLY)==_expected_audio_only_stage2,
      stage2_features_for(METHOD_AUDIO_ONLY))
check('audio_only_stage2_excludes_stage1_score_and_all_cheap',
      'stage1_score' not in stage2_features_for(METHOD_AUDIO_ONLY)
      and not (set(stage2_features_for(METHOD_AUDIO_ONLY)) & set(BASE_FEATURE_NAMES)))
check('cheap_only_disables_stage2',stage2_features_for(METHOD_STAGE1_ONLY)==[])

check('all_taskC_rows',len(ABL_TASKC)==len(GROUPS)*len(ABLATION_METHODS)*len(EVENT_TOLERANCES))
check('all_taskA_rows',len(ABL_TASKA)==len(GROUPS)*len(ABLATION_METHODS))
check('all_taskB_rows',len(ABL_TASKB)==len(GROUPS)*len(ABLATION_METHODS)*len(EVENT_TOLERANCES))
check('taskA_auc_auprc',{'MacroF1','AUROC','AUPRC'}.issubset(ABL_TASKA.columns))
check('taskB_auc_auprc',{'MacroF1','AUROC','AUPRC'}.issubset(ABL_TASKB.columns))
check('finite_taskC',np.isfinite(ABL_TASKC[['P','R','F1']].to_numpy()).all())
check('bounded_taskC',ABL_TASKC[['P','R','F1']].ge(0).all().all() and ABL_TASKC[['P','R','F1']].le(1).all().all())
check('median_localization_reported','localization_median_abs_error_s' in ABL_TASKC.columns)
check('paired_tests_present',len(ABL_TESTS)==(len(ABLATION_METHODS)-1)*(1+2*len(EVENT_TOLERANCES)))
check('all_seven_points_per_taskC_condition',ABL_TASKC.groupby(['method','tolerance']).outer_group.nunique().eq(7).all())
check('all_seven_points_per_paired_test',ABL_PAIRED.groupby(['method','metric','tolerance'],dropna=False).outer_group.nunique().eq(7).all())
check('four_figures_png_pdf',len(FIGURE_FILES)==8 and all(Path(p).is_file() and Path(p).stat().st_size>100 for p in FIGURE_FILES))

_required_contract_cols={
    'method','stage1_remove_cheap','stage2_remove_cheap','stage2_remove_audio',
    'stage2_mode','candidate_discovery_contract','stage1_sha256','stage2_sha256'
}
check('feature_contract_metadata_unambiguous',_required_contract_cols.issubset(ABLATION_FEATURE_CONTRACT.columns),
      sorted(ABLATION_FEATURE_CONTRACT.columns))

# Check expected feature lists globally and for every outer package.
for method in ABLATION_METHODS:
    expected_s1=stage1_features_for(method); expected_s2=stage2_features_for(method)
    row=ABLATION_FEATURE_CONTRACT[ABLATION_FEATURE_CONTRACT.method.eq(method)].iloc[0]
    check(f'{method}::feature_contract_stage1_hash',
          row.stage1_sha256==_support_sha256_json(expected_s1,32),row.stage1_sha256)
    check(f'{method}::feature_contract_stage2_hash',
          row.stage2_sha256==_support_sha256_json(expected_s2,32),row.stage2_sha256)

# Every ablation must reuse the full-configuration hyperparameters and its expected feature list.
for outer in GROUPS:
    sels=[load_outer(outer,m)['selection'] for m in ABLATION_METHODS]
    check(f'{outer}::same_stage1_config',len({int(s['config']) for s in sels})==1,[s['config'] for s in sels])
    check(f'{outer}::same_stage2_C',len({float(s['C']) for s in sels})==1,[s['C'] for s in sels])
    for method in ABLATION_METHODS:
        p=load_outer(outer,method)
        expected_s1=stage1_features_for(method); expected_s2=stage2_features_for(method)
        check(f'{outer}::{method}::exact_stage1_feature_contract',
              list(p['stage1_features'])==list(expected_s1),
              {'actual_n':len(p['stage1_features']),'expected_n':len(expected_s1)})
        check(f'{outer}::{method}::exact_stage2_feature_contract',
              list(p['stage2_features'])==list(expected_s2),
              {'actual_n':len(p['stage2_features']),'expected_n':len(expected_s2)})
        if method==METHOD_AUDIO_ONLY:
            check(f'{outer}::{method}::committed_stage2_audio_only',
                  list(p['stage2_features'])==_expected_audio_only_stage2
                  and 'stage1_score' not in p['stage2_features']
                  and not (set(p['stage2_features']) & set(BASE_FEATURE_NAMES)),
                  p['stage2_features'])
        if method==METHOD_STAGE1_ONLY:
            check(f'{outer}::{method}::committed_stage2_disabled',list(p['stage2_features'])==[])
        check(f'{outer}::{method}::no_train_eval_overlap',outer not in p['train_groups'],p['train_groups'])
        check(f'{outer}::{method}::held_sessions_only',set(p['scores'].session.astype(str)).issubset(set(sessions_for([outer]))))
        path=FOLD_ROOT/str(outer)/(method+'.pkl'); meta=path.with_suffix('.commit.json')
        check(f'{outer}::{method}::commit_hash',
              path.is_file() and meta.is_file() and _sha256_file(path)==json.loads(meta.read_text())['sha256'])

ABLATION_RUNTIME_AUDIT=pd.DataFrame(audit); ABLATION_RUNTIME_AUDIT.to_csv(RESULT_ROOT/'runtime_audit.csv',index=False)
display(ABLATION_RUNTIME_AUDIT)
if not ABLATION_RUNTIME_AUDIT.status.eq('PASS').all():
    raise RuntimeError('Ablation runtime audit failed; do not report this run as complete.')

FINAL_PACKAGE={
    'schema':ABLATION_PROTOCOL,'reporting_revision':ABLATION_REPORTING_REV,'run_fingerprint':RUN_FINGERPRINT,
    'contract':RUN_CONTRACT,'feature_contract':ABLATION_FEATURE_CONTRACT.to_dict('records'),
    'taskC_summary':ABL_TASKC_SUMMARY.to_dict('records'),'taskA_summary':ABL_TASKA_SUMMARY.to_dict('records'),'taskB_summary':ABL_TASKB_SUMMARY.to_dict('records'),
    'article_table':ABLATION_ARTICLE_TABLE.to_dict('records'),'paired_tests':ABL_TESTS.to_dict('records'),
    'main_selection_discovery_log':MAIN_SELECTION_DISCOVERY_LOG,
    'figures':FIGURE_FILES,'audit':audit,
    'interpretation_rule':'Positive Full−Ablated delta means the removed/restricted component helped Full configuration on that metric.',
    'statistical_note':'Paired Wilcoxon tests are exploratory with n=7 outer groups; Holm-adjusted p-values and group-bootstrap confidence intervals are reported.',
    'scope_note':'This is a post-hoc fixed-configuration explanatory ablation. Architecture hyperparameters are selected by Full configuration only; ablations do not perform independent architecture search.'
}
atomic_bytes(RESULT_ROOT/'final_ablation_package.json',canonical_json(FINAL_PACKAGE).encode())
print('FIXED-CONFIGURATION ABLATION AUDIT: PASS')
print('Results:',RESULT_ROOT)

,check,status,detail
0,seven_outer_groups,PASS,"['01-28-2022', '02-11-2022', '03-04-2022', '03..."
1,eight_ablation_conditions,PASS,"('Full configuration', 'Stage-1 only', 'Audio only..."
2,feature_family_partition_complete,PASS,
3,prosody_is_distinct_nonempty_family,PASS,7
4,base_r12_source_parity,PASS,"{'r12_session_table': True, 'R11GeneralAudioAd..."
...,...,...,...
343,12-15-2021::Minus ECAPA::exact_stage1_feat...,PASS,"{'actual_n': 112, 'expected_n': 112}"
344,12-15-2021::Minus ECAPA::exact_stage2_feat...,PASS,"{'actual_n': 257, 'expected_n': 257}"
345,12-15-2021::Minus ECAPA::no_train_eval_ove...,PASS,"['01-28-2022', '02-11-2022', '03-04-2022', '03..."
346,12-15-2021::Minus ECAPA::held_sessions_only,PASS,


R12 FIXED-CONFIGURATION ABLATION AUDIT: PASS
Results: <MULTITPC_ROOT>/runs/MultiTPC_R12_FIXED_CONFIGURATION_ABLATIONS_7FOLD/protocol_f0baefbafdc4b25bb3f0a56be3dcb928/results
